# XACLE 2027 — D01: Expert Score Envelope Analysis (MOE8)

Frozen-checkpoint diagnostic only. This notebook does not train, calibrate, rescale, or alter predictions. It evaluates the public train, validation, and test splits; public test labels, when officially present, are read only after label-free inference and used for descriptive reporting. This is not LOGO or unseen-generator evaluation.

```text
source commit: d9868ba6de085da27074f924e8c15a43d4e3cd1f
profile: full_8 | experts: expert1 … expert8 | score width: 8 | feature width: 33,032
raw scores: [0,10] | normalized targets: (raw - 5) / 5
```

## 1. Runtime and fixed experiment contract

Only the five verified public input datasets below are mounted. The private evaluation data is excluded. All outputs are written under `/kaggle/working/d01_moe8/`.

In [ ]:
import csv, hashlib, json, os, shutil, subprocess, sys, textwrap
from pathlib import Path
import numpy as np

SOURCE_COMMIT = "eb6d8c410e63dcecf3e688dbf22938de95c73bed"
REPO_URL = "https://github.com/vietquang8071/MOESCORE.git"
EXPECTED_COUNTS = {"train": 7500, "validation": 3000, "test": 3000}
D01_ALLOW_PUBLIC_SPLIT_AUDIO_OVERLAP = True
EXPECTED_ORDER = [f"expert{i}" for i in range(1, 9)]
EXPECTED_FEATURE_WIDTH = 33032
ACCURACY_THRESHOLD_RAW = 0.5
SENSITIVITY_THRESHOLDS_RAW = (0.25, 0.5, 1.0)
INPUT_BASE = Path("/kaggle/input/datasets/qnv8071")
INPUTS = {
    "gate": INPUT_BASE / "moe8-bestgate",
    "recovery": INPUT_BASE / "recovery",
    "weights": INPUT_BASE / "xacle-weights",
    "official": INPUT_BASE / "xacle-2027-data",
    "full9": INPUT_BASE / "moescore-full9-feature-cache",
}
DATASET_IDS = {
    "gate": "qnv8071/moe8-bestgate",
    "recovery": "qnv8071/recovery",
    "weights": "qnv8071/xacle-weights",
    "official": "qnv8071/xacle-2027-data",
    "full9": "qnv8071/moescore-full9-feature-cache",
}
WORK = Path("/kaggle/working/d01_moe8")
REPORTS, PREDICTIONS, PLOTS = (WORK / name for name in ("reports", "predictions", "plots"))
REPO = Path("/kaggle/working/MOESCORE-D01")
for directory in (REPORTS, PREDICTIONS, PLOTS):
    directory.mkdir(parents=True, exist_ok=True)
for name, path in INPUTS.items():
    if not path.is_dir():
        raise FileNotFoundError(f"Missing attached {name} dataset {DATASET_IDS[name]} at {path}")
print("Inputs:", json.dumps({k: {"id": DATASET_IDS[k], "path": str(v)} for k, v in INPUTS.items()}, indent=2))
print("Private evaluation inputs: not attached or accessed")

def run_logged(command, log_path, *, cwd):
    log_path = Path(log_path)
    try:
        with log_path.open("w", encoding="utf-8") as log:
            subprocess.run(command, cwd=cwd, stdout=log, stderr=subprocess.STDOUT, check=True, text=True)
    except subprocess.CalledProcessError:
        lines = log_path.read_text(encoding="utf-8", errors="replace").splitlines() if log_path.exists() else []
        print(f"Subprocess failed; last 100 lines of {log_path}:")
        print("\n".join(lines[-100:]))
        raise


## 2. Pinned source and dependency preflight

The existing Kaggle installer preserves the preinstalled PyTorch/CUDA stack. No model inference starts before source and checkpoint preflight.

In [ ]:
if not REPO.exists():
    subprocess.run(["git", "clone", REPO_URL, str(REPO)], check=True)
subprocess.run(["git", "-C", str(REPO), "fetch", "origin"], check=True)
subprocess.run(["git", "-C", str(REPO), "checkout", "--detach", SOURCE_COMMIT], check=True)
actual_commit = subprocess.check_output(["git", "-C", str(REPO), "rev-parse", "HEAD"], text=True).strip()
assert actual_commit == SOURCE_COMMIT, (actual_commit, SOURCE_COMMIT)
required = [
    "model/expert_adapters/registry.py", "model/moe_gating_core.py", "model/moe_forward_integration.py",
    "training/moe_trainer.py", "training/moe_evaluator.py", "training/target_transform.py",
    "data/moe_feature_store.py", "data/full7_assembler.py", "data/full8_assembler.py",
    "data/full8_contract.py", "scripts/build_full7_feature_store.py",
    "scripts/build_full8_feature_store.py", "scripts/extract_expert8_features.py",
    "configs/moe_full8.example.yaml", "configs/expert8_extraction.example.yaml",
]
absent = [name for name in required if not (REPO / name).is_file()]
if absent:
    raise FileNotFoundError(f"Pinned MOESCORE source is incomplete: {absent}")
version = sys.version_info[:2]
if version not in {(3, 12), (3, 13)}:
    raise RuntimeError(f"Unsupported Kaggle Python version for the checked dependency plan: {version}")
requirement_file = REPO / f"requirements-kaggle-py{version[0]}{version[1]}.txt"
subprocess.run([
    sys.executable, str(REPO / "scripts/install_kaggle_dependencies.py"),
    "--requirements", str(requirement_file),
    "--constraints", str(REPORTS / "runtime-constraints.txt"),
    "--report-json", str(REPORTS / "dependency-preflight.json"),
], check=True)
os.chdir(REPO)
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))
print("Pinned source:", actual_commit)

## 3. Diagnostic implementation and public source metadata

The reusable diagnostic module is embedded into the notebook so the Kaggle run uses the exact local implementation even though the upstream commit remains pinned and unchanged.

In [ ]:
DIAGNOSTIC_SOURCE = '"""Frozen MOE8 convex-score-envelope diagnostics; all core inputs are normalized."""\n\nfrom __future__ import annotations\n\nimport csv\nfrom pathlib import Path\nfrom typing import Any, Mapping, Sequence\n\nimport numpy as np\n\n\nRAW_SCORE_MIN = 0.0\nRAW_SCORE_MAX = 10.0\nNORMALIZED_SCORE_SCALE = "normalized_minus1_to1"\nRAW_SCORE_SCALE = "raw_0_to_10"\nDEFAULT_ACCURACY_THRESHOLD_RAW = 0.5\nDEFAULT_SENSITIVITY_THRESHOLDS_RAW = (0.25, 0.5, 1.0)\nSCORE_BINS = tuple(\n    (float(i), float(i + 1), "[9,10]" if i == 9 else f"[{i},{i + 1})")\n    for i in range(10)\n)\nTAIL_BINS = (\n    ("low_tail", 0.0, 3.0, "[0,3)"),\n    ("middle", 3.0, 7.0, "[3,7)"),\n    ("high", 7.0, 9.0, "[7,9)"),\n    ("extreme_high", 9.0, 10.0, "[9,10]"),\n)\nFULL8_INFERENCE_COMPAT_FORMAT_VERSION = 7\nFULL8_INFERENCE_COMPAT_SOURCE_FILES = (\n    "model/moe_gating_core.py", "training/moe_evaluator.py",\n    "training/target_transform.py", "data/moe_feature_store.py",\n    "data/full8_contract.py", "data/full7_assembler.py",\n    "model/expert_adapters/registry.py",\n    "model/expert_adapters/expert8_m2d_clap_adapter.py",\n    "model/expert_adapters/expert8_m2d_clap_runtime.py",\n    "model/expert_adapters/m2d_lora_runtime.py",\n    "scripts/extract_expert8_features.py",\n    "expert3_finetune/XACLE_2027_baselinemodel/models/moescore_expert3.py",\n    "expert3_finetune/XACLE_2027_baselinemodel/models/m2d/vendor/portable_m2d.py",\n)\nFULL8_GATE_ARCHITECTURE = {\n    "input_width": 33032, "hidden_width": 512, "normalization": "LayerNorm",\n    "activation": "ReLU", "dropout": 0.1, "output_width": 8,\n    "output": "Softmax(dim=-1)",\n}\n\n\ndef prepare_format7_gate_payload(\n    payload: Mapping[str, Any], *, current_source_hashes: Mapping[str, str],\n) -> tuple[dict[str, Any], dict[str, Any], dict[str, Any], str]:\n    """Adapt the released Full8-v7 inference metadata to the pinned v5 validator."""\n    if not isinstance(payload, Mapping) or payload.get("format_version") != FULL8_INFERENCE_COMPAT_FORMAT_VERSION:\n        raise ValueError("expected a Full8 format-7 checkpoint payload")\n    if (payload.get("expert_profile") != "full_8"\n            or payload.get("expert_order") != [f"expert{i}" for i in range(1, 9)]\n            or payload.get("expert_feature_dimensions") != {\n                "expert1": 2049, "expert2": 4097, "expert3": 3840, "expert4": 1024,\n                "expert5": 11267, "expert6": 4609, "expert7": 3073, "expert8": 3073,\n            }\n            or payload.get("gate_input_dim") != 33032 or payload.get("gate_expert_count") != 8\n            or payload.get("gate_architecture") != FULL8_GATE_ARCHITECTURE):\n        raise ValueError("Full8 format-7 checkpoint architecture or expert contract is incompatible")\n    recorded_source = payload.get("source_code_sha256")\n    if not isinstance(recorded_source, Mapping) or not recorded_source:\n        raise ValueError("Full8 format-7 checkpoint source-code provenance is missing")\n    for name, digest in recorded_source.items():\n        if (not isinstance(name, str) or not isinstance(digest, str) or len(digest) != 64\n                or any(char not in "0123456789abcdef" for char in digest)):\n            raise ValueError("Full8 format-7 checkpoint source-code provenance is malformed")\n    missing = [name for name in FULL8_INFERENCE_COMPAT_SOURCE_FILES if name not in recorded_source]\n    changed = [name for name in FULL8_INFERENCE_COMPAT_SOURCE_FILES\n               if name in recorded_source and recorded_source[name] != current_source_hashes.get(name)]\n    if missing or changed:\n        raise ValueError(f"Full8 format-7 inference source code is incompatible: missing={missing}, changed={changed}")\n\n    identity = payload.get("feature_store_identity")\n    config = payload.get("config")\n    if not isinstance(identity, Mapping) or not isinstance(config, Mapping):\n        raise ValueError("Full8 format-7 checkpoint identity/configuration is missing")\n    hashes, sample_ids = identity.get("source_wav_sha256_by_split"), identity.get("source_sample_ids_by_split")\n    if (not isinstance(hashes, Mapping) or set(hashes) != {"train", "validation"}\n            or not isinstance(sample_ids, Mapping) or set(sample_ids) != {"train", "validation"}):\n        raise ValueError("Full8 format-7 source audio membership is incomplete")\n    source_rows = {}\n    for split in ("train", "validation"):\n        split_hashes, split_ids = hashes[split], sample_ids[split]\n        if (not isinstance(split_hashes, list) or not isinstance(split_ids, list)\n                or len(split_hashes) != len(split_ids)):\n            raise ValueError(f"Full8 format-7 {split} source audio membership is invalid")\n        source_rows[split] = [\n            {"sample_id": sample_id, "wav_sha256": digest}\n            for sample_id, digest in zip(split_ids, split_hashes)\n        ]\n    from data.audio_overlap import build_train_validation_audio_overlap_diagnostic\n    strict = build_train_validation_audio_overlap_diagnostic(\n        source_rows["train"], source_rows["validation"],\n        allow_train_validation_audio_overlap=False,\n    )\n    allow_overlap = bool(strict["overlapping_sha256_count"])\n    diagnostic = build_train_validation_audio_overlap_diagnostic(\n        source_rows["train"], source_rows["validation"],\n        allow_train_validation_audio_overlap=allow_overlap,\n    )\n    data_config = config.get("data")\n    if not isinstance(data_config, Mapping) or data_config.get("profile") != "full_8":\n        raise ValueError("Full8 format-7 checkpoint configuration is invalid")\n    if ("allow_train_validation_audio_overlap" in data_config\n            and (type(data_config["allow_train_validation_audio_overlap"]) is not bool\n                 or data_config["allow_train_validation_audio_overlap"] != allow_overlap)):\n        raise ValueError("Full8 format-7 overlap policy differs from source membership")\n    if "audio_overlap_diagnostic" in identity:\n        if identity["audio_overlap_diagnostic"] != diagnostic:\n            raise ValueError("Full8 format-7 overlap diagnostic differs from source membership")\n\n    adapted_identity = dict(identity)\n    adapted_identity["audio_overlap_diagnostic"] = diagnostic\n    adapted_config = dict(config)\n    adapted_data_config = dict(data_config)\n    adapted_data_config["allow_train_validation_audio_overlap"] = allow_overlap\n    adapted_config["data"] = adapted_data_config\n    adapted = dict(payload)\n    adapted.update({\n        "format_version": 5,\n        "feature_store_identity": adapted_identity,\n        "config": adapted_config,\n        "source_code_sha256": dict(current_source_hashes),\n    })\n    changed_files = sorted(name for name, digest in recorded_source.items()\n                           if current_source_hashes.get(name) != digest)\n    audit = {\n        "format_version": FULL8_INFERENCE_COMPAT_FORMAT_VERSION,\n        "verified_runtime_files": list(FULL8_INFERENCE_COMPAT_SOURCE_FILES),\n        "different_or_unavailable_files": changed_files,\n        "checkpoint_source_file_count": len(recorded_source),\n        "checkpoint_source_file_missing_from_pinned_tree": sorted(\n            name for name in recorded_source if name not in current_source_hashes),\n        "runtime_files_unrecorded_by_checkpoint": sorted(\n            name for name in FULL8_INFERENCE_COMPAT_SOURCE_FILES if name not in recorded_source),\n    }\n    origin = "reconstructed_from_checkpoint_source_hashes_format7"\n    return adapted, diagnostic, audit, origin\n\n\ndef load_gate_checkpoint_for_d01(\n    checkpoint_path: str | Path, *, gate: Any,\n    expected_expert_checkpoint_provenance: Mapping[str, Any],\n) -> tuple[dict[str, Any], dict[str, Any], dict[str, Any], str]:\n    """Load the frozen gate using pinned validation plus the released v7 adapter."""\n    import torch\n    from training import checkpoint_manager as checkpoint_module\n\n    payload = torch.load(checkpoint_path, map_location="cpu", weights_only=True)\n    if not isinstance(payload, dict):\n        raise ValueError("unsupported gating checkpoint schema")\n    expected_order = [f"expert{i}" for i in range(1, 9)]\n    if (payload.get("expert_profile") != "full_8" or payload.get("expert_order") != expected_order\n            or payload.get("gate_input_dim") != 33032 or payload.get("gate_expert_count") != 8):\n        raise ValueError("attached gate is not the expected eight-expert Full8 checkpoint")\n    manager = checkpoint_module.MoeCheckpointManager()\n    if payload.get("format_version") == checkpoint_module.FULL8_CHECKPOINT_FORMAT_VERSION:\n        loaded = manager.load_for_inference(\n            checkpoint_path, gating_network=gate,\n            expected_expert_checkpoint_provenance=expected_expert_checkpoint_provenance,\n        )\n        current_source = checkpoint_module.full8_source_code_sha256()\n        recorded_source = loaded.get("source_code_sha256", {})\n        audit = {\n            "format_version": loaded.get("format_version"),\n            "verified_runtime_files": list(FULL8_INFERENCE_COMPAT_SOURCE_FILES),\n            "different_or_unavailable_files": sorted(\n                name for name, digest in recorded_source.items() if current_source.get(name) != digest),\n            "checkpoint_source_file_count": len(recorded_source),\n            "checkpoint_source_file_missing_from_pinned_tree": sorted(\n                name for name in recorded_source if name not in current_source),\n            "runtime_files_unrecorded_by_checkpoint": sorted(\n                name for name in FULL8_INFERENCE_COMPAT_SOURCE_FILES if name not in recorded_source),\n        }\n        return loaded, loaded["feature_store_identity"]["audio_overlap_diagnostic"], audit, "checkpoint"\n    if payload.get("format_version") != FULL8_INFERENCE_COMPAT_FORMAT_VERSION:\n        raise ValueError(f"unsupported Full8 gate checkpoint format: {payload.get(\'format_version\')!r}")\n\n    current_source = checkpoint_module.full8_source_code_sha256()\n    adapted, diagnostic, audit, origin = prepare_format7_gate_payload(\n        payload, current_source_hashes=current_source,\n    )\n    checkpoint_module._validate_gate_payload(adapted, gate)\n    expected_provenance = dict(expected_expert_checkpoint_provenance)\n    if payload.get("expert_checkpoint_provenance") != expected_provenance:\n        raise ValueError("gating checkpoint provenance mismatch with the inference feature store")\n    for field, value in (("config", payload.get("config")),\n                         ("feature_store_identity", payload.get("feature_store_identity")),\n                         ("expert_checkpoint_provenance", payload.get("expert_checkpoint_provenance"))):\n        if payload.get(f"{field}_sha256") != checkpoint_module._fingerprint(value, field):\n            raise ValueError(f"checkpoint {field} fingerprint mismatch")\n    state_dict = payload.get("gating_state_dict")\n    if not isinstance(state_dict, dict):\n        raise ValueError("checkpoint has no gating state dictionary")\n    try:\n        gate.load_state_dict(state_dict, strict=True)\n    except RuntimeError as exc:\n        raise ValueError(f"gating state dictionary is incompatible: {exc}") from exc\n    return payload, diagnostic, audit, origin\n\n\ndef _numeric(value: Any, field: str) -> np.ndarray:\n    raw = np.asarray(value)\n    if not np.issubdtype(raw.dtype, np.number) or np.issubdtype(raw.dtype, np.complexfloating):\n        raise TypeError(f"{field} must contain real numeric values")\n    try:\n        result = np.asarray(raw, dtype=np.float64)\n    except (TypeError, ValueError, OverflowError) as exc:\n        raise TypeError(f"{field} must contain real numeric values") from exc\n    if not np.isfinite(result).all():\n        raise ValueError(f"{field} must contain only finite values")\n    return result\n\n\ndef normalize_raw_scores(scores_raw: Any) -> np.ndarray:\n    """Convert human labels from the official raw [0,10] scale to [-1,1]."""\n    raw = _numeric(scores_raw, "scores_raw")\n    if np.any((raw < RAW_SCORE_MIN) | (raw > RAW_SCORE_MAX)):\n        raise ValueError("scores_raw must remain on the original [0,10] scale")\n    return (raw - 5.0) / 5.0\n\n\ndef denormalize_scores(scores_normalized: Any) -> np.ndarray:\n    """Convert normalized values to raw-score units without clamping."""\n    return _numeric(scores_normalized, "scores_normalized") * 5.0 + 5.0\n\n\ndef _validated_inputs(\n    expert_scores_normalized: Any,\n    gate_weights: Any,\n    moe_prediction_normalized: Any | None = None,\n) -> tuple[np.ndarray, np.ndarray, np.ndarray]:\n    scores = _numeric(expert_scores_normalized, "expert_scores_normalized")\n    weights = _numeric(gate_weights, "gate_weights")\n    if scores.ndim != 2 or scores.shape[0] == 0 or scores.shape[1] == 0:\n        raise ValueError("expert_scores_normalized must have shape [N,E] with N,E > 0")\n    if weights.shape != scores.shape:\n        raise ValueError(f"gate_weights must have shape {scores.shape}; got {weights.shape}")\n    if np.any(weights < -1e-7):\n        raise ValueError("gate_weights must be nonnegative")\n    if not np.allclose(weights.sum(axis=1), 1.0, atol=1e-5, rtol=0.0):\n        raise ValueError("gate_weights must sum to one for every sample")\n\n    recomputed = np.einsum("ij,ij->i", scores, weights)\n    if moe_prediction_normalized is None:\n        prediction = recomputed\n    else:\n        prediction = _numeric(moe_prediction_normalized, "moe_prediction_normalized")\n        if prediction.shape != (scores.shape[0],):\n            raise ValueError(f"moe_prediction_normalized must have shape [{scores.shape[0]}]")\n        if not np.allclose(prediction, recomputed, atol=1e-5, rtol=1e-5):\n            raise ValueError("provided MOE predictions differ from weighted expert scores")\n    lower = scores.min(axis=1)\n    upper = scores.max(axis=1)\n    tolerance = 1e-5 * np.maximum(1.0, np.maximum(np.abs(lower), np.abs(upper)))\n    if np.any(prediction < lower - tolerance) or np.any(prediction > upper + tolerance):\n        raise ValueError("convex MOE prediction falls outside the expert score interval")\n    return scores, weights, prediction\n\n\ndef compute_envelope_diagnostics(\n    expert_scores_normalized: Any,\n    gate_weights: Any,\n    moe_prediction_normalized: Any | None = None,\n) -> dict[str, np.ndarray]:\n    """Compute each sample\'s frozen-gate prediction and expert score interval."""\n    scores, weights, prediction = _validated_inputs(\n        expert_scores_normalized, gate_weights, moe_prediction_normalized\n    )\n    lower = scores.min(axis=1)\n    upper = scores.max(axis=1)\n    result = {\n        "expert_scores_normalized": scores,\n        "gate_weights": weights,\n        "expert_min_normalized": lower,\n        "expert_max_normalized": upper,\n        "envelope_width_normalized": upper - lower,\n        "moe_prediction_normalized": prediction,\n        "expert_min_raw": denormalize_scores(lower),\n        "expert_max_raw": denormalize_scores(upper),\n        "envelope_width_raw": (upper - lower) * 5.0,\n        "moe_prediction_raw": denormalize_scores(prediction),\n    }\n    return result\n\n\ndef _empty_summary() -> dict[str, Any]:\n    return {\n        "sample_count": 0,\n        "above_expert_max_count": 0,\n        "above_expert_max_fraction": None,\n        "below_expert_min_count": 0,\n        "below_expert_min_fraction": None,\n        "outside_envelope_count": 0,\n        "outside_envelope_fraction": None,\n        "mean_envelope_width": None,\n        "median_envelope_width": None,\n        "mean_envelope_width_raw": None,\n        "median_envelope_width_raw": None,\n        "mean_envelope_width_normalized": None,\n        "median_envelope_width_normalized": None,\n        "mean_minimum_absolute_error": None,\n        "median_minimum_absolute_error": None,\n        "mean_minimum_absolute_error_raw": None,\n        "median_minimum_absolute_error_raw": None,\n        "mean_minimum_absolute_error_normalized": None,\n        "median_minimum_absolute_error_normalized": None,\n        "convex_mse_lower_bound": None,\n        "convex_mse_lower_bound_raw": None,\n        "convex_mse_lower_bound_normalized": None,\n        "actual_moe_mse": None,\n        "actual_moe_mse_raw": None,\n        "actual_moe_mse_normalized": None,\n        "actual_moe_mse_minus_lower_bound": None,\n        "actual_moe_mse_minus_lower_bound_raw": None,\n        "actual_moe_mse_minus_lower_bound_normalized": None,\n        "mean_signed_error": None,\n        "mean_absolute_error": None,\n        "mean_signed_error_raw": None,\n        "mean_absolute_error_raw": None,\n    }\n\n\ndef _summary_for_indices(envelope: Mapping[str, np.ndarray], labels_norm: np.ndarray, indices: np.ndarray) -> dict[str, Any]:\n    count = int(indices.size)\n    summary = _empty_summary()\n    summary["sample_count"] = count\n    if count == 0:\n        return summary\n\n    lower = envelope["expert_min_normalized"][indices]\n    upper = envelope["expert_max_normalized"][indices]\n    prediction = envelope["moe_prediction_normalized"][indices]\n    target = labels_norm[indices]\n    below = target < lower\n    above = target > upper\n    outside = below | above\n    distance = np.maximum.reduce((lower - target, target - upper, np.zeros(count, dtype=np.float64)))\n    actual_error = prediction - target\n    oracle_mse_norm = float(np.mean(distance**2))\n    actual_mse_norm = float(np.mean(actual_error**2))\n    if actual_mse_norm + max(1e-12, abs(oracle_mse_norm) * 1e-10) < oracle_mse_norm:\n        raise RuntimeError("actual MOE MSE is below the convex-envelope lower bound")\n\n    width = upper - lower\n    summary.update({\n        "above_expert_max_count": int(above.sum()),\n        "above_expert_max_fraction": float(above.mean()),\n        "below_expert_min_count": int(below.sum()),\n        "below_expert_min_fraction": float(below.mean()),\n        "outside_envelope_count": int(outside.sum()),\n        "outside_envelope_fraction": float(outside.mean()),\n        "mean_envelope_width": float(width.mean() * 5.0),\n        "median_envelope_width": float(np.median(width) * 5.0),\n        "mean_envelope_width_raw": float(width.mean() * 5.0),\n        "median_envelope_width_raw": float(np.median(width) * 5.0),\n        "mean_envelope_width_normalized": float(width.mean()),\n        "median_envelope_width_normalized": float(np.median(width)),\n        "mean_minimum_absolute_error": float(distance.mean() * 5.0),\n        "median_minimum_absolute_error": float(np.median(distance) * 5.0),\n        "mean_minimum_absolute_error_raw": float(distance.mean() * 5.0),\n        "median_minimum_absolute_error_raw": float(np.median(distance) * 5.0),\n        "mean_minimum_absolute_error_normalized": float(distance.mean()),\n        "median_minimum_absolute_error_normalized": float(np.median(distance)),\n        "convex_mse_lower_bound": oracle_mse_norm * 25.0,\n        "convex_mse_lower_bound_raw": oracle_mse_norm * 25.0,\n        "convex_mse_lower_bound_normalized": oracle_mse_norm,\n        "actual_moe_mse": actual_mse_norm * 25.0,\n        "actual_moe_mse_raw": actual_mse_norm * 25.0,\n        "actual_moe_mse_normalized": actual_mse_norm,\n        "actual_moe_mse_minus_lower_bound": (actual_mse_norm - oracle_mse_norm) * 25.0,\n        "actual_moe_mse_minus_lower_bound_raw": (actual_mse_norm - oracle_mse_norm) * 25.0,\n        "actual_moe_mse_minus_lower_bound_normalized": actual_mse_norm - oracle_mse_norm,\n        "mean_signed_error": float(actual_error.mean() * 5.0),\n        "mean_absolute_error": float(np.abs(actual_error).mean() * 5.0),\n        "mean_signed_error_raw": float(actual_error.mean() * 5.0),\n        "mean_absolute_error_raw": float(np.abs(actual_error).mean() * 5.0),\n    })\n    return summary\n\n\ndef _score_bin_index(labels_raw: np.ndarray) -> np.ndarray:\n    # side="right" makes each integer boundary start the following bin; score 10 maps to bin 9.\n    return np.searchsorted(np.arange(1.0, 10.0), labels_raw, side="right")\n\n\ndef _named_metrics(summary: Mapping[str, Any], *, label_mean: float | None = None,\n                   moe_mean: float | None = None, lower_mean: float | None = None,\n                   upper_mean: float | None = None) -> dict[str, Any]:\n    return {\n        **dict(summary),\n        "mean_ground_truth": label_mean,\n        "mean_moe_prediction": moe_mean,\n        "mean_expert_min": lower_mean,\n        "mean_expert_max": upper_mean,\n    }\n\n\ndef analyze_split(\n    expert_scores_normalized: Any,\n    gate_weights: Any,\n    moe_prediction_normalized: Any | None = None,\n    *,\n    labels_raw: Any | None = None,\n    accuracy_threshold_raw: float = DEFAULT_ACCURACY_THRESHOLD_RAW,\n    sensitivity_thresholds_raw: Sequence[float] = DEFAULT_SENSITIVITY_THRESHOLDS_RAW,\n) -> dict[str, Any]:\n    """Return row arrays and split, bin, tail, decomposition, and class summaries.\n\n    ``labels_raw=None`` is label-free: no ground-truth, oracle, or error metrics\n    are returned. Classification threshold is fixed in raw 0–10 score units.\n    """\n    envelope = compute_envelope_diagnostics(\n        expert_scores_normalized, gate_weights, moe_prediction_normalized\n    )\n    sample_count = envelope["expert_scores_normalized"].shape[0]\n    result: dict[str, Any] = {\n        "score_scale": NORMALIZED_SCORE_SCALE,\n        "raw_interpretation_scale": RAW_SCORE_SCALE,\n        "sample_count": sample_count,\n        "labels_present": labels_raw is not None,\n        "envelope": envelope,\n        "global_metrics": None,\n        "per_bin_metrics": [],\n        "tail_metrics": [],\n        "structural_vs_gating": None,\n        "classification": None,\n    }\n    if labels_raw is None:\n        return result\n\n    labels = _numeric(labels_raw, "labels_raw")\n    if labels.shape != (sample_count,):\n        raise ValueError(f"labels_raw must have shape [{sample_count}]")\n    labels_norm = normalize_raw_scores(labels)\n    if not np.isfinite(accuracy_threshold_raw) or accuracy_threshold_raw < 0:\n        raise ValueError("accuracy_threshold_raw must be finite and nonnegative")\n    thresholds = _numeric(sensitivity_thresholds_raw, "sensitivity_thresholds_raw").reshape(-1)\n    if thresholds.size == 0 or np.any(thresholds < 0):\n        raise ValueError("sensitivity_thresholds_raw must contain nonnegative values")\n\n    above = labels_norm > envelope["expert_max_normalized"]\n    below = labels_norm < envelope["expert_min_normalized"]\n    outside = above | below\n    distance = np.maximum.reduce((\n        envelope["expert_min_normalized"] - labels_norm,\n        labels_norm - envelope["expert_max_normalized"],\n        np.zeros(sample_count, dtype=np.float64),\n    ))\n    actual_error_norm = envelope["moe_prediction_normalized"] - labels_norm\n    actual_sq_norm = actual_error_norm**2\n    oracle_sq_norm = distance**2\n    difference_norm = actual_sq_norm - oracle_sq_norm\n    if np.any(difference_norm < -1e-10):\n        raise RuntimeError("sample-wise actual squared error is below its convex oracle lower bound")\n    envelope.update({\n        "label_raw": labels,\n        "label_normalized": labels_norm,\n        "oracle_projection_normalized": np.clip(\n            labels_norm, envelope["expert_min_normalized"], envelope["expert_max_normalized"]\n        ),\n        "oracle_projection_raw": denormalize_scores(np.clip(\n            labels_norm, envelope["expert_min_normalized"], envelope["expert_max_normalized"]\n        )),\n        "above_expert_max": above,\n        "below_expert_min": below,\n        "outside_envelope": outside,\n        "minimum_absolute_error_normalized": distance,\n        "minimum_absolute_error_raw": distance * 5.0,\n        "actual_gate_squared_error_normalized": actual_sq_norm,\n        "oracle_convex_squared_error_normalized": oracle_sq_norm,\n        "actual_minus_oracle_squared_error_normalized": difference_norm,\n        "actual_gate_squared_error_raw": actual_sq_norm * 25.0,\n        "oracle_convex_squared_error_raw": oracle_sq_norm * 25.0,\n        "actual_minus_oracle_squared_error_raw": difference_norm * 25.0,\n    })\n\n    all_indices = np.arange(sample_count)\n    global_summary = _summary_for_indices(envelope, labels_norm, all_indices)\n    result["global_metrics"] = global_summary\n    bin_indices = _score_bin_index(labels)\n    per_bin = []\n    for index, (start, end, name) in enumerate(SCORE_BINS):\n        selected = all_indices[bin_indices == index]\n        summary = _summary_for_indices(envelope, labels_norm, selected)\n        mean = lambda field: float(np.mean(envelope[field][selected])) if selected.size else None\n        per_bin.append({\n            "ground_truth_bin": name,\n            **_named_metrics(\n                summary,\n                label_mean=float(np.mean(labels[selected])) if selected.size else None,\n                moe_mean=mean("moe_prediction_raw"),\n                lower_mean=mean("expert_min_raw"),\n                upper_mean=mean("expert_max_raw"),\n            ),\n            "class_A_count": int(np.sum((selected.size > 0) & above[selected])) if selected.size else 0,\n            "class_B_count": int(np.sum((selected.size > 0) & below[selected])) if selected.size else 0,\n        })\n    result["per_bin_metrics"] = per_bin\n\n    tails = []\n    for name, start, end, label in TAIL_BINS:\n        selected = all_indices[(labels >= start) & ((labels <= end) if end == 10.0 else (labels < end))]\n        summary = _summary_for_indices(envelope, labels_norm, selected)\n        tails.append({\n            "tail": name, "ground_truth_range": label,\n            **_named_metrics(\n                summary,\n                label_mean=float(np.mean(labels[selected])) if selected.size else None,\n                moe_mean=float(np.mean(envelope["moe_prediction_raw"][selected])) if selected.size else None,\n                lower_mean=float(np.mean(envelope["expert_min_raw"][selected])) if selected.size else None,\n                upper_mean=float(np.mean(envelope["expert_max_raw"][selected])) if selected.size else None,\n            ),\n        })\n    result["tail_metrics"] = tails\n    result["structural_vs_gating"] = {\n        "score_scale_normalized": NORMALIZED_SCORE_SCALE,\n        "raw_interpretation_scale": RAW_SCORE_SCALE,\n        "actual_gate_squared_error_mean_normalized": float(actual_sq_norm.mean()),\n        "oracle_convex_squared_error_mean_normalized": float(oracle_sq_norm.mean()),\n        "difference_mean_normalized": float(difference_norm.mean()),\n        "actual_gate_squared_error_mean_raw": float(actual_sq_norm.mean() * 25.0),\n        "oracle_convex_squared_error_mean_raw": float(oracle_sq_norm.mean() * 25.0),\n        "difference_mean_raw": float(difference_norm.mean() * 25.0),\n        "by_score_bin": [\n            {\n                "ground_truth_bin": row["ground_truth_bin"],\n                "sample_count": row["sample_count"],\n                "actual_gate_squared_error_mean_raw": row["actual_moe_mse"],\n                "oracle_convex_squared_error_mean_raw": row["convex_mse_lower_bound"],\n                "difference_mean_raw": row["actual_moe_mse_minus_lower_bound"],\n                "actual_gate_squared_error_mean_normalized": row["actual_moe_mse_normalized"],\n                "oracle_convex_squared_error_mean_normalized": row["convex_mse_lower_bound_normalized"],\n                "difference_mean_normalized": row["actual_moe_mse_minus_lower_bound_normalized"],\n            }\n            for row in per_bin\n        ],\n    }\n\n    signed_error_raw = actual_error_norm * 5.0\n    classifications = np.full(sample_count, "D", dtype="U1")\n    inside = ~outside\n    classifications[inside & (np.abs(signed_error_raw) > accuracy_threshold_raw)] = "C"\n    classifications[above] = "A"\n    classifications[below] = "B"\n    class_summary = {\n        label: {\n            "sample_count": int(np.sum(classifications == label)),\n            "fraction": float(np.mean(classifications == label)),\n        }\n        for label in ("A", "B", "C", "D")\n    }\n    sensitivity = []\n    for threshold in sorted(set(float(value) for value in thresholds) | {float(accuracy_threshold_raw)}):\n        classes = np.full(sample_count, "D", dtype="U1")\n        classes[inside & (np.abs(signed_error_raw) > threshold)] = "C"\n        classes[above] = "A"\n        classes[below] = "B"\n        sensitivity.append({\n            "accuracy_threshold_raw": threshold,\n            "classification_counts": {label: int(np.sum(classes == label)) for label in ("A", "B", "C", "D")},\n        })\n    result["classification"] = {\n        "accuracy_threshold_raw": float(accuracy_threshold_raw),\n        "class_summary": class_summary,\n        "sensitivity_check": sensitivity,\n        "per_sample": classifications.tolist(),\n    }\n    return result\n\n\ndef summarize_by_generator(\n    analysis: Mapping[str, Any],\n    *,\n    generator_ids: Sequence[Any] | None,\n    generator_names: Sequence[Any] | None = None,\n) -> list[dict[str, Any]]:\n    """Summarize label-based envelope metrics by official generator metadata."""\n    if not analysis.get("labels_present"):\n        return [{"status": "LABELS_UNAVAILABLE"}]\n    if generator_ids is None and generator_names is None:\n        return [{"status": "GENERATOR_METADATA_UNAVAILABLE"}]\n    ids = generator_ids if generator_ids is not None else generator_names\n    if len(ids) != analysis["sample_count"]:\n        raise ValueError("generator metadata length must match feature-store order")\n    if any(value is None or not str(value).strip() for value in ids):\n        raise ValueError("generator IDs/names must be present for every joined sample")\n    names = generator_names\n    if names is not None and len(names) != len(ids):\n        raise ValueError("generator_names length must match generator_ids")\n    envelope = analysis["envelope"]\n    labels_norm = envelope["label_normalized"]\n    groups: dict[str, list[int]] = {}\n    display_names: dict[str, str] = {}\n    for index, value in enumerate(ids):\n        key = str(value).strip()\n        groups.setdefault(key, []).append(index)\n        if names is not None:\n            display_names.setdefault(key, str(names[index]).strip())\n    rows = []\n    for key in sorted(groups):\n        selected = np.asarray(groups[key], dtype=np.int64)\n        summary = _summary_for_indices(envelope, labels_norm, selected)\n        rows.append({\n            "status": "AVAILABLE",\n            "generator_id": key,\n            "generator_name": display_names.get(key, ""),\n            "sample_count": summary["sample_count"],\n            "outside_envelope_fraction": summary["outside_envelope_fraction"],\n            "above_max_fraction": summary["above_expert_max_fraction"],\n            "below_min_fraction": summary["below_expert_min_fraction"],\n            "convex_mse_lower_bound": summary["convex_mse_lower_bound"],\n            "convex_mse_lower_bound_normalized": summary["convex_mse_lower_bound_normalized"],\n            "actual_moe_mse": summary["actual_moe_mse"],\n            "actual_moe_mse_normalized": summary["actual_moe_mse_normalized"],\n            "mean_envelope_width": summary["mean_envelope_width"],\n            "mean_envelope_width_normalized": summary["mean_envelope_width_normalized"],\n        })\n    return rows\n\n\ndef build_prediction_rows(\n    sample_ids: Sequence[str],\n    split: str,\n    expert_scores_normalized: Any,\n    gate_weights: Any,\n    analysis: Mapping[str, Any],\n    *,\n    labels_raw: Any | None = None,\n    model_ids: Sequence[Any] | None = None,\n    model_names: Sequence[Any] | None = None,\n) -> list[dict[str, Any]]:\n    """Build ordered, CSV-ready rows without changing or calibrating scores."""\n    if split not in {"train", "validation", "test"}:\n        raise ValueError("split must be train, validation, or test")\n    ids = list(sample_ids)\n    scores, weights, prediction = _validated_inputs(\n        expert_scores_normalized, gate_weights, analysis["envelope"]["moe_prediction_normalized"]\n    )\n    count = scores.shape[0]\n    if len(ids) != count or any(not isinstance(value, str) or not value for value in ids):\n        raise ValueError("sample_ids must be non-empty strings in feature-store order")\n    if len(ids) != len(set(ids)):\n        raise ValueError("sample_ids must be unique")\n    if any(not sample_id.startswith(split + ":") for sample_id in ids):\n        raise ValueError("sample_ids do not match the requested split")\n    labels = None if labels_raw is None else _numeric(labels_raw, "labels_raw")\n    if labels is not None and (labels.shape != (count,) or np.any((labels < 0) | (labels > 10))):\n        raise ValueError("labels_raw must be finite [N] on the original [0,10] scale")\n    for values, name in ((model_ids, "model_ids"), (model_names, "model_names")):\n        if values is not None and len(values) != count:\n            raise ValueError(f"{name} length must match sample_ids")\n    if labels is None and analysis.get("labels_present"):\n        raise ValueError("analysis includes labels but prediction rows were asked to omit them")\n    if labels is not None and not analysis.get("labels_present"):\n        raise ValueError("prediction labels were provided but analysis did not include them")\n    classes = analysis.get("classification", {}).get("per_sample") if labels is not None else None\n    envelope = analysis["envelope"]\n    rows: list[dict[str, Any]] = []\n    for i, sample_id in enumerate(ids):\n        row: dict[str, Any] = {\n            "sample_id": sample_id,\n            "split": split,\n            "label_raw": "" if labels is None else float(labels[i]),\n        }\n        if model_ids is not None:\n            row["model_id"] = str(model_ids[i])\n        if model_names is not None:\n            row["model_name"] = str(model_names[i])\n        for expert_index in range(scores.shape[1]):\n            expert_name = f"expert{expert_index + 1}"\n            row[f"{expert_name}_score_normalized"] = float(scores[i, expert_index])\n            row[f"{expert_name}_score_raw"] = float(denormalize_scores(scores[i, expert_index]))\n            row[f"{expert_name}_gate_weight"] = float(weights[i, expert_index])\n        row.update({\n            "moe_prediction_normalized": float(prediction[i]),\n            "moe_prediction_raw": float(envelope["moe_prediction_raw"][i]),\n            "expert_min_normalized": float(envelope["expert_min_normalized"][i]),\n            "expert_max_normalized": float(envelope["expert_max_normalized"][i]),\n            "expert_min_raw": float(envelope["expert_min_raw"][i]),\n            "expert_max_raw": float(envelope["expert_max_raw"][i]),\n            "envelope_width_normalized": float(envelope["envelope_width_normalized"][i]),\n            "envelope_width_raw": float(envelope["envelope_width_raw"][i]),\n        })\n        if labels is not None:\n            for key in (\n                "above_expert_max", "below_expert_min", "outside_envelope",\n                "actual_gate_squared_error_normalized", "oracle_convex_squared_error_normalized",\n                "actual_minus_oracle_squared_error_normalized", "actual_gate_squared_error_raw",\n                "oracle_convex_squared_error_raw", "actual_minus_oracle_squared_error_raw",\n            ):\n                row[key] = (bool(envelope[key][i]) if key in {"above_expert_max", "below_expert_min", "outside_envelope"}\n                            else float(envelope[key][i]))\n            row["convex_class"] = classes[i]\n            row["oracle_projection_normalized"] = float(envelope["oracle_projection_normalized"][i])\n            row["oracle_projection_raw"] = float(envelope["oracle_projection_raw"][i])\n        rows.append(row)\n    return rows\n\n\ndef write_prediction_csv(path: str | Path, rows: Sequence[Mapping[str, Any]]) -> None:\n    """Write prediction rows in the supplied sample order."""\n    if not rows:\n        raise ValueError("cannot write an empty prediction file")\n    fields = list(rows[0])\n    if any(list(row) != fields for row in rows):\n        raise ValueError("prediction rows must have identical ordered columns")\n    target = Path(path)\n    target.parent.mkdir(parents=True, exist_ok=True)\n    with target.open("w", encoding="utf-8", newline="") as stream:\n        writer = csv.DictWriter(stream, fieldnames=fields, extrasaction="raise")\n        writer.writeheader()\n        writer.writerows(rows)\n\n\ndef validate_d01_feature_store(\n    root: str | Path,\n    profile: str,\n    expected_counts: Mapping[str, int],\n    *,\n    source_manifest_root: str | Path,\n) -> dict[str, dict[str, Any]]:\n    """Validate each public split against its own canonical source manifest."""\n    import hashlib\n\n    from data.moe_feature_store import preprocessing_contract, validate_feature_store\n    from data.set_a_cached_dataset import _read_source_manifest\n    from model.expert_adapters.registry import profile_contract\n\n    if profile not in {"full_7", "full_8"}:\n        raise ValueError(f"D01 profile must be full_7 or full_8, got {profile!r}")\n    splits = ("train", "validation", "test")\n    if set(expected_counts) != set(splits) or any(\n        type(expected_counts[split]) is not int or expected_counts[split] <= 0 for split in splits\n    ):\n        raise ValueError("D01 expected_counts must give positive integer counts for train/validation/test")\n\n    order, dimensions, width = profile_contract(profile)\n    source_root = Path(source_manifest_root)\n    result: dict[str, dict[str, Any]] = {}\n    reference_provenance = None\n    reference_preprocessing = None\n    for split in splits:\n        context = {"root": str(root), "split": split, "profile": profile}\n        try:\n            manifest = validate_feature_store(root, split=split, require_labels=split != "test")\n            if manifest.get("split") != split:\n                raise ValueError("manifest split identity mismatch")\n            if manifest.get("expert_profile") != profile:\n                raise ValueError(f"expert profile mismatch: {manifest.get(\'expert_profile\')!r}")\n            if (manifest.get("expert_order") != list(order)\n                    or manifest.get("feature_dimensions") != dimensions\n                    or manifest.get("feature_width") != width\n                    or manifest.get("score_width") != len(order)):\n                raise ValueError("expert order/count or feature dimensions do not match the profile contract")\n            rows = manifest["shards"]\n            sample_ids = [sample_id for shard in rows for sample_id in shard["sample_ids"]]\n            filenames = [name for shard in rows for name in shard["wave_file_names"]]\n            sample_count = sum(shard["count"] for shard in rows)\n            if sample_count != expected_counts[split]:\n                raise ValueError(f"sample count {sample_count} != expected {expected_counts[split]}")\n            if len(sample_ids) != len(set(sample_ids)) or len(filenames) != len(set(filenames)):\n                raise ValueError("sample IDs or source filenames are not unique")\n            if manifest.get("labels_present") is not (split != "test"):\n                raise ValueError("label presence does not match the public split contract")\n\n            source_digest = manifest.get("source_manifest_sha256")\n            if (not isinstance(source_digest, str) or len(source_digest) != 64\n                    or any(character not in "0123456789abcdef" for character in source_digest.lower())):\n                raise ValueError("source manifest digest is not a 64-character hexadecimal SHA-256")\n            source_records = _read_source_manifest(\n                source_root / split / "source_manifest.json",\n                split=split,\n                expected_sha256=source_digest,\n            )\n            if len(source_records) != expected_counts[split]:\n                raise ValueError("source manifest row count differs from the expected split count")\n            if ([record["sample_id"] for record in source_records] != sample_ids\n                    or [record["wave_file_name"] for record in source_records] != filenames):\n                raise ValueError("source manifest sample IDs/filenames differ from feature-store order")\n            fingerprints = manifest["assembly_identity"]["ordered_source_fingerprints"]\n            observed_fingerprints = [\n                {\n                    "sample_id": record["sample_id"],\n                    "wav_sha256": record["wav_sha256"].lower(),\n                    "caption_sha256": hashlib.sha256(record["raw_caption"].encode("utf-8")).hexdigest(),\n                }\n                for record in source_records\n            ]\n            if fingerprints != observed_fingerprints:\n                raise ValueError("source manifest WAV/caption fingerprints differ from feature-store assembly")\n\n            provenance = manifest["checkpoint_provenance"]\n            preprocessing = preprocessing_contract(manifest)\n            if reference_provenance is None:\n                reference_provenance, reference_preprocessing = provenance, preprocessing\n            elif provenance != reference_provenance:\n                raise ValueError("expert checkpoint provenance differs across public splits")\n            elif preprocessing != reference_preprocessing:\n                raise ValueError("compatible preprocessing contracts differ across public splits")\n            result[split] = manifest\n        except Exception as exc:\n            raise ValueError(f"{split}: D01 feature-store validation failed ({context}): {exc}") from exc\n    return result\n\n\n__all__ = [\n    "DEFAULT_ACCURACY_THRESHOLD_RAW",\n    "DEFAULT_SENSITIVITY_THRESHOLDS_RAW",\n    "FULL8_INFERENCE_COMPAT_FORMAT_VERSION",\n    "FULL8_INFERENCE_COMPAT_SOURCE_FILES",\n    "FULL8_GATE_ARCHITECTURE",\n    "NORMALIZED_SCORE_SCALE",\n    "RAW_SCORE_SCALE",\n    "SCORE_BINS",\n    "TAIL_BINS",\n    "analyze_split",\n    "build_prediction_rows",\n    "compute_envelope_diagnostics",\n    "denormalize_scores",\n    "load_gate_checkpoint_for_d01",\n    "normalize_raw_scores",\n    "prepare_format7_gate_payload",\n    "summarize_by_generator",\n    "validate_d01_feature_store",\n    "write_prediction_csv",\n]\n'
diagnostic_path = REPO / "scripts/diagnostics/d01_moe8.py"
diagnostic_path.parent.mkdir(parents=True, exist_ok=True)
(diagnostic_path.parent / "__init__.py").write_text("", encoding="utf-8")
diagnostic_path.write_text(DIAGNOSTIC_SOURCE, encoding="utf-8")
from scripts.diagnostics.d01_moe8 import (
    analyze_split, build_prediction_rows, summarize_by_generator, write_prediction_csv,
)
from data.moe_feature_store import validate_feature_store, preprocessing_contract
from data.moe_training_dataset import MoeTrainingDataset, collate_moe_features
from model.expert_adapters.registry import FULL8_EXPERT_ORDER, profile_contract
from model.moe_gating_core import GatingNetwork
from training.checkpoint_manager import MoeCheckpointManager
from training.target_transform import normalize_targets
from training.moe_evaluator import MoeGateEvaluator
import torch
assert tuple(FULL8_EXPERT_ORDER) == tuple(EXPECTED_ORDER)
order, dimensions, width = profile_contract("full_8")
assert list(order) == EXPECTED_ORDER and len(dimensions) == 8 and width == EXPECTED_FEATURE_WIDTH
assert np.array_equal(normalize_targets(torch.tensor([0.0, 5.0, 10.0])).numpy(), np.array([-1.0, 0.0, 1.0]))
print({"profile": "full_8", "expert_order": list(order), "expert_dimensions": dimensions, "feature_width": width})


## 4. Dataset and checkpoint discovery

Paths are resolved only under the five explicitly attached public datasets. Official average-score CSV row order defines sample order.

In [ ]:
def file_sha256(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as stream:
        for chunk in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

def unique_file(root, filename):
    matches = sorted(p for p in root.rglob(filename) if p.is_file())
    hashes = {file_sha256(p) for p in matches}
    if not matches:
        raise FileNotFoundError(f"{filename} not found under verified dataset {root}")
    if len(hashes) != 1:
        raise ValueError(f"Ambiguous {filename} files with different contents under {root}: {matches}")
    return matches[0]

gate_checkpoint = unique_file(INPUTS["gate"], "gate8_best.pt")
gate_sha256 = file_sha256(gate_checkpoint)
data_candidates = []
for path in INPUTS["official"].rglob("train_average.csv"):
    candidate = path.parent.parent
    if path.parent.name == "meta_data" and all((candidate / "meta_data" / f"{s}_average.csv").is_file()
                                                 and (candidate / "wav" / s).is_dir()
                                                 for s in EXPECTED_COUNTS):
        data_candidates.append(candidate)
data_candidates = sorted(set(data_candidates))
if len(data_candidates) != 1:
    raise ValueError(f"Expected one official public dataset root, found {data_candidates}")
DATA_ROOT = data_candidates[0]

def read_public_metadata(split, *, read_labels):
    path = DATA_ROOT / "meta_data" / f"{split}_average.csv"
    with path.open("r", newline="", encoding="utf-8-sig") as stream:
        reader = csv.DictReader(stream)
        headers = {h.strip().lower(): h for h in (reader.fieldnames or [])}
        filename_key = next((headers[k] for k in ("wav_file_name", "wave_file_name") if k in headers), None)
        caption_key = headers.get("text")
        score_key = next((headers[k] for k in ("average_score", "average_semantic_alignement_score",
                                                   "average_semantic_alignment_score") if k in headers), None)
        id_key, name_key = headers.get("model_id"), headers.get("model_name")
        if not filename_key or not caption_key:
            raise ValueError(f"Official {split} CSV lacks filename/caption columns")
        if read_labels and score_key is None:
            raise ValueError(f"Official {split} CSV does not contain released labels")
        rows = []
        for raw in reader:
            filename, caption = (raw.get(filename_key) or "").strip(), raw.get(caption_key)
            if not filename or Path(filename).name != filename or not caption or not caption.strip():
                raise ValueError(f"Invalid source row in {path}")
            row = {"sample_id": f"{split}:{filename}", "wave_file_name": filename, "raw_caption": caption,
                   "model_id": (raw.get(id_key, "").strip() if id_key else ""),
                   "model_name": (raw.get(name_key, "").strip() if name_key else "")}
            if read_labels:
                row["label_raw"] = float(raw[score_key])
                if not np.isfinite(row["label_raw"]) or not 0 <= row["label_raw"] <= 10:
                    raise ValueError(f"Invalid released score in {path}")
            rows.append(row)
    if len(rows) != EXPECTED_COUNTS[split] or len({r["sample_id"] for r in rows}) != len(rows):
        raise ValueError(f"{split}: expected {EXPECTED_COUNTS[split]} unique rows, found {len(rows)}")
    return rows

metadata = {s: read_public_metadata(s, read_labels=s != "test") for s in EXPECTED_COUNTS}
for split, rows in metadata.items():
    assert len(rows) == EXPECTED_COUNTS[split]
    assert (DATA_ROOT / "wav" / split).is_dir()
print("Public split counts:", {s: len(v) for s, v in metadata.items()})
print("Gate checkpoint:", str(gate_checkpoint), gate_sha256)

## 5. Reuse or reconstruct a compatible Full8 store

The notebook first validates attached Full8 stores against the frozen gate. Otherwise it reuses a verified Full7 store or builds Full7 from the Full9 and Whisper caches plus the released E5/E6 heads, then appends Expert8 using the repository's resumable extraction and assembly scripts.

In [ ]:
from data.audio_overlap import (
    build_public_split_audio_overlap_diagnostic,
    build_train_validation_audio_overlap_diagnostic,
    validate_public_split_audio_overlap_diagnostic,
)
from data.set_a_cached_dataset import _read_source_manifest
from scripts.kaggle_full9_artifacts import restore_feature_cache, validate_full9_store
from scripts.infer_moe_gating import _reject_test_wav_overlap
from data.moe_feature_store import validate_feature_store
from training.moe_evaluator import MoeGateEvaluator
from model.moe_gating_core import GatingNetwork
from training.checkpoint_manager import FULL8_CHECKPOINT_FORMAT_VERSION
from scripts.diagnostics.d01_moe8 import (
    FULL8_INFERENCE_COMPAT_FORMAT_VERSION,
    load_gate_checkpoint_for_d01,
    validate_d01_feature_store,
)

untrusted_gate_payload = torch.load(gate_checkpoint, map_location="cpu", weights_only=True)
if (not isinstance(untrusted_gate_payload, dict)
        or untrusted_gate_payload.get("format_version") not in {
            FULL8_CHECKPOINT_FORMAT_VERSION, FULL8_INFERENCE_COMPAT_FORMAT_VERSION}
        or untrusted_gate_payload.get("expert_profile") != "full_8"
        or untrusted_gate_payload.get("expert_order") != EXPECTED_ORDER
        or untrusted_gate_payload.get("gate_input_dim") != 33032
        or untrusted_gate_payload.get("gate_expert_count") != 8):
    raise ValueError("Attached gate is not a supported eight-expert Full8 checkpoint")
preflight_gate = GatingNetwork(input_feature_dim=33032, num_experts=8, profile="full_8")
(preflight_payload, EXPECTED_OVERLAP_DIAGNOSTIC, GATE_SOURCE_CODE_AUDIT,
 GATE_OVERLAP_POLICY_ORIGIN) = load_gate_checkpoint_for_d01(
    gate_checkpoint, gate=preflight_gate,
    expected_expert_checkpoint_provenance=untrusted_gate_payload["expert_checkpoint_provenance"])
preflight_gate.eval()
assert preflight_payload["gate_architecture"] == {
    "input_width": 33032, "hidden_width": 512, "normalization": "LayerNorm",
    "activation": "ReLU", "dropout": 0.1, "output_width": 8, "output": "Softmax(dim=-1)"}
del preflight_gate, preflight_payload, untrusted_gate_payload

FULL9_ROOT = None

def discover_full9_source_root():
    def candidates(base):
        found = set()
        if base.exists():
            for manifest_path in base.rglob("train/manifest.json"):
                body = json.loads(manifest_path.read_text(encoding="utf-8"))
                if body.get("expert_profile") == "full_9":
                    found.add(manifest_path.parent.parent.resolve())
        return sorted(found)

    found = candidates(INPUTS["full9"])
    restored_root = WORK / "full9_restored"
    restored_store = restored_root / "feature_cache"
    if not found and restored_store.is_dir():
        found = candidates(restored_store)
    if not found:
        upload_manifests = list(INPUTS["full9"].rglob("upload_manifest.json"))
        if len(upload_manifests) != 1:
            raise FileNotFoundError(
                "Full9 cache needs one direct verified store or exactly one upload_manifest.json; "
                f"found {len(upload_manifests)} upload manifests"
            )
        restore_feature_cache(upload_manifests[0].parent, restored_root)
        found = candidates(restored_store)
    if len(found) != 1:
        raise ValueError(f"Expected one Full9 source store for per-split manifest verification, found {found}")
    return found[0]

def store_candidates(profile):
    found = set()
    for root in INPUTS.values():
        for manifest_path in root.rglob("train/manifest.json"):
            candidate = manifest_path.parent.parent.resolve()
            try:
                body = json.loads(manifest_path.read_text(encoding="utf-8"))
                if body.get("expert_profile") == profile and all((candidate / s / "manifest.json").is_file()
                                                                   for s in EXPECTED_COUNTS):
                    found.add(candidate)
            except (OSError, ValueError) as exc:
                print(f"Ignoring unreadable feature-store candidate {manifest_path}: {type(exc).__name__}: {exc}")
    return sorted(found)

def validate_store(root, profile):
    global FULL9_ROOT
    if FULL9_ROOT is None:
        FULL9_ROOT = discover_full9_source_root()
    try:
        return validate_d01_feature_store(
            root, profile, EXPECTED_COUNTS, source_manifest_root=FULL9_ROOT)
    except Exception as exc:
        print(f"D01 {profile} validation failed for {root}; expected per-split counts={EXPECTED_COUNTS}")
        print(f"Failing split/invariant: {type(exc).__name__}: {exc}")
        for split in EXPECTED_COUNTS:
            manifest_path = Path(root) / split / "manifest.json"
            if manifest_path.is_file():
                try:
                    details = json.loads(manifest_path.read_text(encoding="utf-8"))
                except (OSError, ValueError) as metadata_exc:
                    print(f"{split} identity metadata unreadable: {type(metadata_exc).__name__}: {metadata_exc}")
                else:
                    print(f"{split} store identity:", {
                        key: details.get(key) for key in
                        ("split", "expert_profile", "source_manifest_sha256", "store_identity_sha256")
                    })
        raise

D01_PUBLIC_SPLIT_OVERLAP_DIAGNOSTIC = None

def record_public_split_overlap(records_by_split, *, source):
    global D01_PUBLIC_SPLIT_OVERLAP_DIAGNOSTIC
    for split, expected in EXPECTED_COUNTS.items():
        actual_ids = [row["sample_id"] for row in records_by_split[split]]
        official_ids = [row["sample_id"] for row in metadata[split]]
        if len(actual_ids) != expected or actual_ids != official_ids:
            raise ValueError(f"{source}: {split} sample IDs/count/order differ from official public metadata")
    diagnostic = build_public_split_audio_overlap_diagnostic(
        records_by_split,
        allow_public_split_audio_overlap=D01_ALLOW_PUBLIC_SPLIT_AUDIO_OVERLAP,
        allow_train_validation_audio_overlap=True,
    )
    validate_public_split_audio_overlap_diagnostic(diagnostic)
    if diagnostic["status"] == "ERROR":
        raise ValueError(f"D01 public overlap policy unexpectedly rejected source rows: {diagnostic}")
    artifact = dict(diagnostic)
    artifact.update({
        "source": source,
        "active_diagnostic_override_policy": {
            "allow_public_split_audio_overlap": D01_ALLOW_PUBLIC_SPLIT_AUDIO_OVERLAP,
            "scope": "D01 public train/validation/test feature-store diagnostics only",
        },
        "public_sample_count_by_split": dict(EXPECTED_COUNTS),
        "train_validation_overlap_count": diagnostic["pair_overlap_group_counts"]["train_validation"],
        "train_test_overlap_count": diagnostic["pair_overlap_group_counts"]["train_test"],
        "validation_test_overlap_count": diagnostic["pair_overlap_group_counts"]["validation_test"],
        "retained_public_sample_count": sum(EXPECTED_COUNTS.values()),
        "all_public_samples_retained": True,
        "evaluation_warning": "Cross-split byte-identical WAV duplicates may bias evaluation; no rows were removed or reordered.",
    })
    (REPORTS / "audio_overlap.json").write_text(json.dumps(artifact, indent=2, allow_nan=False) + "\n")
    D01_PUBLIC_SPLIT_OVERLAP_DIAGNOSTIC = diagnostic
    if diagnostic["total_overlapping_sha256_groups"]:
        print("WARNING: cross-split public WAV duplicates may bias evaluation:",
              diagnostic["pair_overlap_group_counts"])
    return diagnostic

def manifest_source_rows(manifests):
    return {split: manifests[split]["assembly_identity"]["ordered_source_fingerprints"]
            for split in EXPECTED_COUNTS}

def gate_probe(root):
    manifests = validate_store(root, "full_8")
    datasets = {s: MoeTrainingDataset(str(root), split=s, require_labels=s != "test") for s in EXPECTED_COUNTS}
    gate = GatingNetwork(input_feature_dim=33032, num_experts=8, profile="full_8")
    payload, observed_gate_overlap, _, _ = load_gate_checkpoint_for_d01(
        gate_checkpoint, gate=gate,
        expected_expert_checkpoint_provenance=manifests["test"]["checkpoint_provenance"])
    if observed_gate_overlap != EXPECTED_OVERLAP_DIAGNOSTIC:
        raise ValueError("Frozen gate overlap diagnostic changed between preflight and store validation")
    identity = payload["feature_store_identity"]
    if identity["train"] != manifests["train"]["store_identity_sha256"] or identity["validation"] != manifests["validation"]["store_identity_sha256"]:
        raise ValueError("Full8 train/validation store identities differ from the gate checkpoint")
    observed_overlap = build_train_validation_audio_overlap_diagnostic(
        manifests["train"]["assembly_identity"]["ordered_source_fingerprints"],
        manifests["validation"]["assembly_identity"]["ordered_source_fingerprints"],
        allow_train_validation_audio_overlap=EXPECTED_OVERLAP_DIAGNOSTIC["allow_train_validation_audio_overlap"])
    if observed_overlap != EXPECTED_OVERLAP_DIAGNOSTIC:
        raise ValueError("Full8 train/validation overlap diagnostic differs from gate checkpoint source hashes")
    expected_preprocess = identity["preprocessing_contract"]
    if any(preprocessing_contract(manifests[s]) != expected_preprocess for s in EXPECTED_COUNTS):
        raise ValueError("Full8 preprocessing contract differs from gate checkpoint")
    public_diagnostic = record_public_split_overlap(manifest_source_rows(manifests), source=f"Full8 store {root}")
    if not D01_ALLOW_PUBLIC_SPLIT_AUDIO_OVERLAP:
        _reject_test_wav_overlap(identity, manifests["test"])
    gate.eval()
    return manifests, datasets, gate, payload

FULL8_ROOT = None
FULL8_MANIFESTS = FULL8_DATASETS = GATE = GATE_PAYLOAD = None
FULL8_CANDIDATE_ERRORS = []
for candidate in store_candidates("full_8"):
    try:
        FULL8_MANIFESTS, FULL8_DATASETS, GATE, GATE_PAYLOAD = gate_probe(candidate)
        FULL8_ROOT = candidate
        print("Reusing provenance-compatible Full8 store:", FULL8_ROOT)
        break
    except Exception as exc:
        FULL8_CANDIDATE_ERRORS.append({"path": str(candidate), "error": f"{type(exc).__name__}: {exc}"})
print("Rejected Full8 candidates:", FULL8_CANDIDATE_ERRORS)
if FULL8_ROOT is not None:
    (REPORTS / "full7_build.json").write_text(json.dumps({
        "status": "SKIPPED", "reason": "compatible Full8 store reused; Full7 reconstruction was unnecessary",
        "retained_public_sample_count": sum(EXPECTED_COUNTS.values()), "all_public_samples_retained": True,
    }, indent=2) + "\n")
    (REPORTS / "full7_build.log").write_text("Skipped: provenance-compatible Full8 store reused.\n", encoding="utf-8")

FULL7_ROOT = None
FULL7_MANIFESTS = None
if FULL8_ROOT is None:
    for candidate in store_candidates("full_7"):
        try:
            FULL7_MANIFESTS = validate_store(candidate, "full_7")
            FULL7_ROOT = candidate
            print("Reusing verified Full7 store:", FULL7_ROOT)
            break
        except Exception as exc:
            print("Ignoring incompatible Full7 candidate:", candidate, f"{type(exc).__name__}: {exc}")
    if FULL7_ROOT is not None:
        record_public_split_overlap(manifest_source_rows(FULL7_MANIFESTS), source=f"reused Full7 store {FULL7_ROOT}")
        (REPORTS / "full7_build.json").write_text(json.dumps({
            "status": "REUSED", "full7_store_root": str(FULL7_ROOT),
            "public_split_audio_overlap_diagnostic": D01_PUBLIC_SPLIT_OVERLAP_DIAGNOSTIC,
            "retained_public_sample_count": sum(EXPECTED_COUNTS.values()), "all_public_samples_retained": True,
        }, indent=2) + "\n")
        (REPORTS / "full7_build.log").write_text(f"Reused verified Full7 store: {FULL7_ROOT}\n", encoding="utf-8")

    if FULL7_ROOT is None:
        if FULL9_ROOT is None:
            FULL9_ROOT = discover_full9_source_root()
        validate_full9_store(FULL9_ROOT, EXPECTED_COUNTS)
        E5_CHECKPOINT = unique_file(INPUTS["recovery"], "expert5_best.pt")
        E6_CHECKPOINT = unique_file(INPUTS["recovery"], "expert6_best.pt")
        WHISPER_ROOT = INPUTS["recovery"] / "recovery/whisper_cache"
        for split, count in EXPECTED_COUNTS.items():
            if not (WHISPER_ROOT / split / "manifest.json").is_file():
                raise FileNotFoundError(f"Missing verified Whisper cache manifest: {WHISPER_ROOT / split}")
        source_records = {}
        for split in EXPECTED_COUNTS:
            manifest = validate_feature_store(FULL9_ROOT, split=split, require_labels=split != "test")
            source_records[split] = _read_source_manifest(
                FULL9_ROOT / split / "source_manifest.json", split=split,
                expected_sha256=manifest["source_manifest_sha256"])
        overlap = build_train_validation_audio_overlap_diagnostic(
            source_records["train"], source_records["validation"],
            allow_train_validation_audio_overlap=EXPECTED_OVERLAP_DIAGNOSTIC["allow_train_validation_audio_overlap"])
        if overlap != EXPECTED_OVERLAP_DIAGNOSTIC:
            raise ValueError("Full9 source audio overlap differs from the frozen gate's training provenance")
        record_public_split_overlap(source_records, source=f"Full9 source store {FULL9_ROOT}")
        FULL7_ROOT = WORK / "full7_store"
        FULL7_ROOT.mkdir(parents=True, exist_ok=True)
        full7_cmd = [sys.executable, str(REPO / "scripts/build_full7_feature_store.py"),
                     "--full9-root", str(FULL9_ROOT), "--whisper-root", str(WHISPER_ROOT),
                     "--e5-checkpoint", str(E5_CHECKPOINT), "--e6-checkpoint", str(E6_CHECKPOINT),
                     "--source-audio-root", str(DATA_ROOT / "wav"), "--output-root", str(FULL7_ROOT),
                     "--report-json", str(REPORTS / "full7_build.json")]
        for split in EXPECTED_COUNTS:
            full7_cmd += ["--split", split]
        if D01_ALLOW_PUBLIC_SPLIT_AUDIO_OVERLAP:
            full7_cmd += ["--allow-train-validation-audio-overlap", "--allow-public-split-audio-overlap"]
        run_logged(full7_cmd, REPORTS / "full7_build.log", cwd=REPO)
        full7_build = json.loads((REPORTS / "full7_build.json").read_text(encoding="utf-8"))
        if full7_build.get("status") != "PASS":
            raise RuntimeError(f"Full7 builder did not report PASS: {full7_build}")
        FULL7_MANIFESTS = validate_store(FULL7_ROOT, "full_7")
        rebuilt_diagnostic = record_public_split_overlap(
            manifest_source_rows(FULL7_MANIFESTS), source=f"built Full7 store {FULL7_ROOT}")
        if full7_build.get("public_split_audio_overlap_diagnostic") != rebuilt_diagnostic:
            raise ValueError("Full7 build overlap report differs from the verified output manifests")
        print("Full7 built from attached Full9, Whisper and E5/E6 caches; all 13,500 public rows retained.")

## 6. Expert8 preflight and Full8 assembly, only when needed

The existing Expert8 extraction script is resumable and validates source row identities. The gate is loaded only after a complete Full8 store is available.

In [ ]:
if FULL8_ROOT is None:
    import yaml
    from model.expert_adapters.expert8_m2d_clap_runtime import (
        load_expert8_checkpoint, validate_base_m2d_checkpoint,
    )
    from data.moe_expert_cache import load_expert_shard, validate_expert_cache_identity
    from scripts.extract_expert8_features import (
        CORAL_FEATURE_DIM, _shard_sample_identity,
        validate_expert8_cache_manifest,
    )
    E8_LORA = unique_file(INPUTS["weights"], "m2d-lora-Nam.pt")
    E8_BASE = unique_file(INPUTS["weights"], "checkpoint-30.pth")
    lora_sha256 = file_sha256(E8_LORA)
    base_sha256 = validate_base_m2d_checkpoint(E8_BASE)
    e8_payload = load_expert8_checkpoint(E8_LORA)
    assert len(e8_payload["lora"]) == 24 and tuple(e8_payload["head"]["net.5.weight"].shape) == (10, 256)
    CONFIG_ROOT = WORK / "configs"
    CONFIG_ROOT.mkdir(parents=True, exist_ok=True)
    HF_CACHE = WORK / "hf_e8_cache"
    os.environ["HF_HOME"] = str(HF_CACHE)
    os.environ["HF_HUB_CACHE"] = str(HF_CACHE / "hub")
    e8_config = yaml.safe_load((REPO / "configs/expert8_extraction.example.yaml").read_text())
    e8_config["runtime"].update({"device": "cuda:0", "offline": False, "cache_dir": str(HF_CACHE)})
    e8_config["experts"]["expert8"]["base_checkpoint"] = str(E8_BASE)
    e8_config["experts"]["expert8"]["finetuned_checkpoint"] = str(E8_LORA)
    E8_CONFIG = CONFIG_ROOT / "expert8_extraction.yaml"
    E8_CONFIG.write_text(yaml.safe_dump(e8_config, sort_keys=False))
    E8_DATA_ROOT = WORK / "expert8_label_free_source"
    E8_META_ROOT = E8_DATA_ROOT / "meta_data"
    E8_META_ROOT.mkdir(parents=True, exist_ok=True)
    if not (E8_DATA_ROOT / "wav").exists():
        (E8_DATA_ROOT / "wav").symlink_to(DATA_ROOT / "wav", target_is_directory=True)
    for split in ("train", "validation"):
        source_csv = DATA_ROOT / "meta_data" / f"{split}_average.csv"
        target_csv = E8_META_ROOT / source_csv.name
        if not target_csv.exists():
            target_csv.symlink_to(source_csv)
    test_source_csv = DATA_ROOT / "meta_data/test_average.csv"
    test_target_csv = E8_META_ROOT / "test_average.csv"
    score_aliases = {"average_score", "average_semantic_alignement_score", "average_semantic_alignment_score"}
    if not test_target_csv.exists():
        with test_source_csv.open("r", newline="", encoding="utf-8-sig") as source_stream:
            reader = csv.DictReader(source_stream)
            source_fields = list(reader.fieldnames or [])
            selected_score_fields = [name for name in source_fields if name.strip().lower() in score_aliases]
            if len(selected_score_fields) > 1:
                raise ValueError("Public test CSV has ambiguous score columns")
            label_free_fields = [name for name in source_fields if name not in selected_score_fields]
            with test_target_csv.open("w", newline="", encoding="utf-8") as target_stream:
                writer = csv.DictWriter(target_stream, fieldnames=label_free_fields)
                writer.writeheader()
                for row in reader:
                    writer.writerow({name: row[name] for name in label_free_fields})
    with test_target_csv.open("r", newline="", encoding="utf-8-sig") as stream:
        test_header = {name.strip().lower() for name in (csv.DictReader(stream).fieldnames or [])}
    if test_header & score_aliases:
        raise ValueError("Expert8 extraction view must be label-free for public test")
    source_rows = {}
    for split in EXPECTED_COUNTS:
        fingerprints = FULL7_MANIFESTS[split]["assembly_identity"]["ordered_source_fingerprints"]
        if len(fingerprints) != len(metadata[split]):
            raise ValueError(f"{split}: Full7 fingerprints do not align with official rows")
        rows = []
        for fingerprint, official in zip(fingerprints, metadata[split]):
            caption_sha = hashlib.sha256(official["raw_caption"].encode("utf-8")).hexdigest()
            if fingerprint["sample_id"] != official["sample_id"] or fingerprint["caption_sha256"] != caption_sha:
                raise ValueError(f"{split}: Full7 source identity differs from official caption/order")
            rows.append({"sample_id": official["sample_id"], "wave_file_name": official["wave_file_name"],
                         "raw_caption": official["raw_caption"], "wav_sha256": fingerprint["wav_sha256"]})
        source_rows[split] = rows
    cache_roots = list(INPUTS.values())
    restored_cache = WORK / "full9_restored/feature_cache"
    if restored_cache.is_dir():
        cache_roots.append(restored_cache)

    def validate_e8_cache(cache_root, cache_id):
        cache_root = Path(cache_root)
        cache_identity = None
        for split in EXPECTED_COUNTS:
            manifest_path = cache_root / "expert8" / cache_id / split / "manifest.json"
            if not manifest_path.is_file():
                raise FileNotFoundError(manifest_path)
            manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
            identity = manifest.get("expert_cache_identity")
            validate_expert_cache_identity(identity)
            provenance = identity.get("checkpoint_provenance", {})
            if (provenance.get("base_checkpoint", {}).get("sha256") != base_sha256
                    or provenance.get("finetuned_checkpoint", {}).get("sha256") != lora_sha256):
                raise ValueError("Expert8 cache checkpoint identities differ from attached weights")
            if cache_identity is None:
                cache_identity = identity
            elif identity != cache_identity:
                raise ValueError("Expert8 split cache identities differ")
            validate_expert8_cache_manifest(manifest, split=split,
                                            expected_records=source_rows[split],
                                            cache_identity=cache_identity)
            cache_base = manifest_path.parent
            for item in manifest["shards"]:
                selected = source_rows[split][item["start"]:item["start"] + item["count"]]
                sample_identity = _shard_sample_identity(split, selected, cache_identity)
                shard_path = cache_base / item["relative_path"]
                load_expert_shard(shard_path, expert_identity=cache_identity,
                                  sample_identity=sample_identity, feature_dim=CORAL_FEATURE_DIM)
        return cache_identity

    def assemble_and_probe(cache_root, cache_id, tag):
        output = WORK / f"full8_{tag}"
        command = [sys.executable, str(REPO / "scripts/build_full8_feature_store.py"),
                   "--full7-root", str(FULL7_ROOT), "--expert8-cache-root", str(cache_root),
                   "--expert8-cache-id", cache_id, "--output-root", str(output),
                   "--report-json", str(REPORTS / f"full8_{tag}_build.json")]
        for split in EXPECTED_COUNTS:
            command += ["--split", split]
        if D01_ALLOW_PUBLIC_SPLIT_AUDIO_OVERLAP:
            command += ["--allow-train-validation-audio-overlap", "--allow-public-split-audio-overlap"]
        full8_log = REPORTS / f"full8_{tag}_build.log"
        run_logged(command, full8_log, cwd=REPO)
        full8_report = json.loads((REPORTS / f"full8_{tag}_build.json").read_text(encoding="utf-8"))
        if full8_report.get("status") != "PASS":
            raise RuntimeError(f"Full8 builder did not report PASS: {full8_report}")
        if full8_report.get("public_split_audio_overlap_diagnostic") != D01_PUBLIC_SPLIT_OVERLAP_DIAGNOSTIC:
            raise ValueError("Full8 builder overlap report differs from the verified Full7 source")
        manifests, datasets, gate, payload = gate_probe(output)
        return output, manifests, datasets, gate, payload

    e8_candidates = set()
    for root in cache_roots:
        for manifest_path in root.rglob("expert8/*/train/manifest.json"):
            candidate_root = manifest_path.parents[3]
            candidate_id = manifest_path.parents[1].name
            e8_candidates.add((candidate_root.resolve(), candidate_id))
    accepted_cache = False
    cache_rejections = []
    for candidate_root, candidate_id in sorted(e8_candidates, key=lambda v: (str(v[0]), v[1])):
        try:
            identity = validate_e8_cache(candidate_root, candidate_id)
            if identity["cache_id"] != candidate_id:
                raise ValueError("Expert8 cache directory name differs from its content identity")
            FULL8_ROOT, FULL8_MANIFESTS, FULL8_DATASETS, GATE, GATE_PAYLOAD = assemble_and_probe(
                candidate_root, candidate_id, f"reused_{candidate_id[:10]}")
            accepted_cache = True
            print("Reused provenance-compatible Expert8 cache:", candidate_root, candidate_id)
            break
        except Exception as exc:
            cache_rejections.append({"cache_root": str(candidate_root), "cache_id": candidate_id,
                                     "error": f"{type(exc).__name__}: {exc}"})
    (REPORTS / "expert8_cache_candidates.json").write_text(
        json.dumps({"accepted": accepted_cache, "rejected": cache_rejections}, indent=2) + "\n")

    if not accepted_cache:
        if not torch.cuda.is_available():
            raise RuntimeError("No checkpoint-compatible Expert8 cache exists; this run needs its configured T4 GPU")
        if "T4" not in torch.cuda.get_device_name(0):
            raise RuntimeError(f"Expected the configured Kaggle T4 for missing Expert8 features; got {torch.cuda.get_device_name(0)}")
        E8_CACHE_ROOT = WORK / "expert8_cache"
        E8_CACHE_ROOT.mkdir(parents=True, exist_ok=True)
        e8_report_path = REPORTS / "expert8_extraction.json"
        e8_cmd = [sys.executable, str(REPO / "scripts/extract_expert8_features.py"),
                  "--config", str(E8_CONFIG), "--dataset-root", str(E8_DATA_ROOT),
                  "--output-root", str(E8_CACHE_ROOT), "--device", "cuda:0",
                  "--batch-size", "1", "--shard-size", "8", "--report-json", str(e8_report_path)]
        run_logged(e8_cmd, REPORTS / "expert8_extraction.log", cwd=REPO)
        e8_report = json.loads(e8_report_path.read_text())
        if (e8_report.get("status") != "PASS" or e8_report.get("lora_checkpoint_sha256") != lora_sha256
                or e8_report.get("base_m2d_checkpoint_sha256") != base_sha256):
            raise ValueError("Expert8 cache did not validate against the attached frozen checkpoints")
        FULL8_ROOT, FULL8_MANIFESTS, FULL8_DATASETS, GATE, GATE_PAYLOAD = assemble_and_probe(
            E8_CACHE_ROOT, e8_report["expert_cache_identity"]["cache_id"], "extracted")
        print("Extracted only Expert8 outputs and assembled Full8.")
    else:
        print("Attached Expert8 cache reused; no Expert8 GPU inference required.")
else:
    print("Full8 reused; no Expert8 GPU inference required.")
assert FULL8_MANIFESTS["train"]["feature_width"] == EXPECTED_FEATURE_WIDTH
assert FULL8_MANIFESTS["test"]["labels_present"] is False
print("Full8 root:", FULL8_ROOT)

## 7. Frozen MOE8 inference and output alignment

The gate uses the verified `full_8` Linear → LayerNorm → ReLU → Dropout → Linear → Softmax architecture. Inference runs on CPU when features are cached; Expert8 extraction is the only GPU inference stage.

In [ ]:
DEVICE = torch.device("cpu")
GATE = GATE.to(DEVICE)
GATE.eval()
assert not GATE.training
evaluator = MoeGateEvaluator(GATE)
split_outputs = {}
for split, dataset in FULL8_DATASETS.items():
    loader = torch.utils.data.DataLoader(dataset, batch_size=128, shuffle=False, num_workers=0,
                                         collate_fn=collate_moe_features)
    ids, scores_rows, weight_rows, predictions, store_labels = [], [], [], [], []
    with torch.inference_mode():
        for batch in loader:
            prediction, weights, scores, features = evaluator._batch_outputs(batch)
            assert scores.shape[1] == 8 and weights.shape == scores.shape
            ids.extend(batch["sample_ids"])
            scores_rows.append(scores.cpu().numpy())
            weight_rows.append(weights.cpu().numpy())
            predictions.append(prediction.cpu().numpy())
            if "labels_raw" in batch:
                store_labels.append(batch["labels_raw"].cpu().numpy())
    scores = np.concatenate(scores_rows).astype(np.float64)
    weights = np.concatenate(weight_rows).astype(np.float64)
    pred = np.concatenate(predictions).astype(np.float64)
    assert ids == [row["sample_id"] for row in metadata[split]], f"{split}: sample ordering mismatch"
    assert scores.shape == (EXPECTED_COUNTS[split], 8) and weights.shape == scores.shape
    assert np.isfinite(scores).all() and np.isfinite(weights).all() and np.isfinite(pred).all()
    assert np.allclose(weights.sum(axis=1), 1.0, atol=1e-5)
    assert (weights >= -1e-7).all()
    recomputed = np.einsum("ij,ij->i", scores, weights)
    assert np.allclose(pred, recomputed, atol=1e-6, rtol=1e-6)
    labels_store = np.concatenate(store_labels) if store_labels else None
    if split != "test":
        official_labels = np.asarray([r["label_raw"] for r in metadata[split]], dtype=np.float64)
        if labels_store is None or not np.allclose(labels_store, official_labels, atol=1e-6):
            raise ValueError(f"{split}: Full8 labels differ from the official average CSV")
    else:
        assert labels_store is None, "Public test feature store must remain label-free"
    analysis = analyze_split(scores, weights, pred, labels_raw=labels_store,
                             accuracy_threshold_raw=ACCURACY_THRESHOLD_RAW,
                             sensitivity_thresholds_raw=SENSITIVITY_THRESHOLDS_RAW)
    split_outputs[split] = {"ids": ids, "scores": scores, "weights": weights,
                            "prediction": pred, "labels_store": labels_store, "analysis": analysis}
    print(split, "inference PASS", len(ids), "mean envelope width raw", float(analysis["envelope"]["envelope_width_raw"].mean()))

# Independent invocation of the repository's label-free reference path on public test only.
from scripts.infer_moe_gating import infer_checkpoint
import scripts.infer_moe_gating as inference_module
from data.audio_overlap import build_public_split_audio_overlap_diagnostic
reference_path = REPORTS / "repository_test_inference.csv"
reference_config = {"runtime": {"device": "cpu"},
                    "data": {"feature_store_root": str(FULL8_ROOT), "profile": "full_8"},
                    "inference": {"batch_size": 128}}
import training.checkpoint_manager as checkpoint_module
from scripts.diagnostics.d01_moe8 import prepare_format7_gate_payload
pinned_gate_validator = checkpoint_module._validate_gate_payload
def _d01_format7_validator(payload, gating_network):
    if payload.get("format_version") == FULL8_INFERENCE_COMPAT_FORMAT_VERSION:
        adapted, _, _, _ = prepare_format7_gate_payload(
            payload, current_source_hashes=checkpoint_module.full8_source_code_sha256())
        return pinned_gate_validator(adapted, gating_network)
    return pinned_gate_validator(payload, gating_network)
checkpoint_module._validate_gate_payload = _d01_format7_validator
pinned_overlap_rejector = inference_module._reject_test_wav_overlap
def _d01_public_overlap_check(identity, test_manifest):
    if not D01_ALLOW_PUBLIC_SPLIT_AUDIO_OVERLAP:
        return pinned_overlap_rejector(identity, test_manifest)
    saved_ids = identity.get("source_sample_ids_by_split")
    saved_hashes = identity.get("source_wav_sha256_by_split")
    if not isinstance(saved_ids, dict) or not isinstance(saved_hashes, dict):
        raise ValueError("Frozen checkpoint lacks train/validation source membership for D01 overlap audit")
    rows = {}
    for split in ("train", "validation"):
        if split not in saved_ids or split not in saved_hashes or len(saved_ids[split]) != len(saved_hashes[split]):
            raise ValueError(f"Frozen checkpoint {split} source membership is malformed")
        rows[split] = [{"sample_id": sample_id, "wav_sha256": digest}
                       for sample_id, digest in zip(saved_ids[split], saved_hashes[split])]
    rows["test"] = test_manifest["assembly_identity"]["ordered_source_fingerprints"]
    observed = build_public_split_audio_overlap_diagnostic(
        rows, allow_public_split_audio_overlap=True, allow_train_validation_audio_overlap=True)
    if observed != D01_PUBLIC_SPLIT_OVERLAP_DIAGNOSTIC:
        raise ValueError("Reference inference public split overlap differs from verified D01 stores")
inference_module._reject_test_wav_overlap = _d01_public_overlap_check
try:
    with torch.inference_mode():
        reference = infer_checkpoint(reference_config, str(gate_checkpoint), str(reference_path), device_override="cpu")
finally:
    checkpoint_module._validate_gate_payload = pinned_gate_validator
    inference_module._reject_test_wav_overlap = pinned_overlap_rejector
with reference_path.open("r", newline="", encoding="utf-8") as stream:
    reference_rows = list(csv.DictReader(stream))
assert reference["status"] == "PASS"
ref_scores = np.asarray([float(row["pred_score"]) for row in reference_rows], dtype=np.float64)
ours_raw = split_outputs["test"]["analysis"]["envelope"]["moe_prediction_raw"]
assert len(reference_rows) == len(ours_raw) == EXPECTED_COUNTS["test"]
assert [f"test:{row['wave_file_name']}" for row in reference_rows] == split_outputs["test"]["ids"]
assert np.allclose(ref_scores, np.round(ours_raw, 2), atol=1e-8, rtol=0.0)
print("Gate prediction equivalence: repository label-free test inference matches rounded D01 outputs")

## 8. Post-inference public test labels and D01 metrics

Train results are descriptive and may be in-sample. Validation results are held-out diagnostics but inherit the project’s historical validation-selection choices. Test labels are joined only now, after all label-free test predictions have been frozen.

In [ ]:
test_rows_after_inference = read_public_metadata("test", read_labels=True) if any(
    key.strip().lower() in {"average_score", "average_semantic_alignement_score", "average_semantic_alignment_score"}
    for key in next(csv.reader((DATA_ROOT / "meta_data/test_average.csv").open(encoding="utf-8-sig")))
) else None
if test_rows_after_inference is not None:
    assert [r["sample_id"] for r in test_rows_after_inference] == split_outputs["test"]["ids"]
    test_labels = np.asarray([r["label_raw"] for r in test_rows_after_inference], dtype=np.float64)
    split_outputs["test"]["analysis"] = analyze_split(
        split_outputs["test"]["scores"], split_outputs["test"]["weights"],
        split_outputs["test"]["prediction"], labels_raw=test_labels,
        accuracy_threshold_raw=ACCURACY_THRESHOLD_RAW,
        sensitivity_thresholds_raw=SENSITIVITY_THRESHOLDS_RAW)
    split_outputs["test"]["labels_official"] = test_labels
    print("Released public test labels used post-inference for descriptive metrics only.")
else:
    split_outputs["test"]["labels_official"] = None
    print("Official public test labels unavailable; test remains label-free in D01 metrics.")

global_rows, bin_rows, generator_rows = [], [], []
generator_available = False
structural = {}
for split, data in split_outputs.items():
    labels = data["analysis"]["envelope"].get("label_raw")
    if labels is None:
        labels = data.get("labels_official")
    analysis = data["analysis"]
    if labels is None:
        global_rows.append({"split": split, "status": "LABELS_UNAVAILABLE", "sample_count": len(data["ids"])})
        continue
    metrics = {"split": split, **analysis["global_metrics"]}
    global_rows.append(metrics)
    bin_rows.extend({"split": split, **row} for row in analysis["per_bin_metrics"])
    structural[split] = analysis["structural_vs_gating"]
    official_rows = metadata[split] if split != "test" else test_rows_after_inference
    id_field = "model_id" if official_rows and all(r.get("model_id") for r in official_rows) else None
    name_field = "model_name" if official_rows and all(r.get("model_name") for r in official_rows) else None
    if id_field or name_field:
        rows = summarize_by_generator(analysis,
            generator_ids=[r[id_field] for r in official_rows] if id_field else None,
            generator_names=[r[name_field] for r in official_rows] if name_field else None)
        generator_available = generator_available or bool(rows and rows[0].get("status") == "AVAILABLE")
        generator_rows.extend({"split": split, **row} for row in rows)
    else:
        generator_rows.append({"split": split, "status": "GENERATOR_METADATA_UNAVAILABLE", "sample_count": len(data["ids"])})

# Optional sensitivity is metric-only: original feature stores and prediction rows retain every sample.
overlap_sensitivity = {}
for split, data in split_outputs.items():
    labels = data["analysis"]["envelope"].get("label_raw")
    affected_ids = set(D01_PUBLIC_SPLIT_OVERLAP_DIAGNOSTIC["affected_sample_ids_by_split"][split])
    keep = np.asarray([sample_id not in affected_ids for sample_id in data["ids"]], dtype=bool)
    item = {
        "split": split,
        "overlapping_sample_count": int((~keep).sum()),
        "non_overlapping_sample_count": int(keep.sum()),
        "all_samples_retained_in_predictions": True,
    }
    if labels is None:
        item["status"] = "LABELS_UNAVAILABLE"
    elif not keep.any():
        item["status"] = "NO_NON_OVERLAPPING_SAMPLES"
    else:
        item["status"] = "AVAILABLE"
        item["all_sample_metrics"] = data["analysis"]["global_metrics"]
        item["non_overlapping_metrics"] = analyze_split(
            data["scores"][keep], data["weights"][keep], data["prediction"][keep],
            labels_raw=np.asarray(labels)[keep], accuracy_threshold_raw=ACCURACY_THRESHOLD_RAW,
            sensitivity_thresholds_raw=SENSITIVITY_THRESHOLDS_RAW,
        )["global_metrics"]
    overlap_sensitivity[split] = item
(REPORTS / "overlap_metric_sensitivity.json").write_text(
    json.dumps(overlap_sensitivity, indent=2, allow_nan=False) + "\n")


## 9. Prediction exports and reports

Every CSV preserves feature-store sample order and carries the eight expert scores and eight frozen gate weights in separate columns. Test labels are blank when unavailable. No embedding tensors are exported.

In [ ]:
from scripts.diagnostics.d01_moe8 import build_prediction_rows, write_prediction_csv

for split, data in split_outputs.items():
    analysis = data["analysis"]
    labels = analysis["envelope"].get("label_raw")
    if labels is None:
        labels = data.get("labels_official")
        if labels is not None:
            analysis = analyze_split(data["scores"], data["weights"], data["prediction"], labels_raw=labels,
                                     accuracy_threshold_raw=ACCURACY_THRESHOLD_RAW,
                                     sensitivity_thresholds_raw=SENSITIVITY_THRESHOLDS_RAW)
    meta_rows = metadata[split] if split != "test" else (test_rows_after_inference or metadata[split])
    model_ids = [r["model_id"] for r in meta_rows] if meta_rows and all(r.get("model_id") for r in meta_rows) else None
    model_names = [r["model_name"] for r in meta_rows] if meta_rows and all(r.get("model_name") for r in meta_rows) else None
    rows = build_prediction_rows(data["ids"], split, data["scores"], data["weights"], analysis,
                                 labels_raw=labels, model_ids=model_ids, model_names=model_names)
    write_prediction_csv(PREDICTIONS / f"{split}_d01.csv", rows)
    data["analysis"] = analysis

def write_dict_csv(path, rows):
    if not rows:
        path.write_text("status\nNO_ROWS\n", encoding="utf-8")
        return
    columns = list(dict.fromkeys(key for row in rows for key in row))
    with path.open("w", newline="", encoding="utf-8") as stream:
        writer = csv.DictWriter(stream, fieldnames=columns, extrasaction="ignore")
        writer.writeheader()
        writer.writerows(rows)

write_dict_csv(REPORTS / "global_metrics.csv", global_rows)
write_dict_csv(REPORTS / "per_bin_metrics.csv", bin_rows)
if not generator_rows:
    generator_rows = [{"status": "GENERATOR_METADATA_UNAVAILABLE"}]
write_dict_csv(REPORTS / "generator_metrics.csv", generator_rows)
(REPORTS / "structural_vs_gating.json").write_text(json.dumps(structural, indent=2, allow_nan=False) + "\n")
score_contracts = FULL8_MANIFESTS["train"]["assembly_identity"]["score_contracts"]
score_scale_audit = {}
for expert_index, expert_name in enumerate(EXPECTED_ORDER):
    score_scale_audit[expert_name] = {"contract": score_contracts[expert_name], "by_split": {}}
    for split, output in split_outputs.items():
        values = output["scores"][:, expert_index]
        raw_interpretation = values * 5.0 + 5.0
        score_scale_audit[expert_name]["by_split"][split] = {
            "sample_count": int(values.size), "normalized_min": float(values.min()),
            "normalized_max": float(values.max()), "normalized_mean": float(values.mean()),
            "normalized_std": float(values.std()),
            "fraction_outside_minus1_to1": float(np.mean((values < -1.0) | (values > 1.0))),
            "raw_interpretation_min": float(raw_interpretation.min()),
            "raw_interpretation_max": float(raw_interpretation.max()),
            "fraction_outside_raw_0_to10_display": float(np.mean((raw_interpretation < 0) | (raw_interpretation > 10))),
        }
summary = {
    "experiment": "XACLE 2027 D01 Expert Score Envelope Analysis",
    "source_commit": SOURCE_COMMIT, "score_scale": "raw_0_to_10; core envelope calculations normalized_minus1_to1",
    "profile": "full_8", "expert_order": EXPECTED_ORDER, "feature_width": EXPECTED_FEATURE_WIDTH,
    "split_counts": {s: len(v["ids"]) for s, v in split_outputs.items()},
    "retained_public_sample_count": sum(len(v["ids"]) for v in split_outputs.values()),
    "all_public_samples_retained": sum(len(v["ids"]) for v in split_outputs.values()) == 13500,
    "public_split_audio_overlap_diagnostic": D01_PUBLIC_SPLIT_OVERLAP_DIAGNOSTIC,
    "overlap_metric_sensitivity": overlap_sensitivity,
    "labels_present": {s: bool(v["analysis"]["labels_present"]) for s, v in split_outputs.items()},
    "metrics_by_split": {s: v["analysis"]["global_metrics"] for s, v in split_outputs.items()},
    "classification_by_split": {s: v["analysis"]["classification"] for s, v in split_outputs.items()},
    "train_is_in_sample": True, "not_logo": True,
    "generator_metadata_status": "AVAILABLE" if generator_available else "GENERATOR_METADATA_UNAVAILABLE",
    "test_labels_accessed_after_inference": test_rows_after_inference is not None,
    "expert_score_scale_audit": score_scale_audit,
}
(REPORTS / "summary.json").write_text(json.dumps(summary, indent=2, allow_nan=False) + "\n")

## 10. High-resolution figures

Plots use raw-score axes from 0 to 10 and matched split samples. Raw transformed expert proxies outside that display interval remain unchanged in CSVs and metrics; the figures note the omitted display tails rather than clipping or recalibrating values.

In [ ]:
import matplotlib.pyplot as plt
def save_plot(fig, filename):
    fig.tight_layout()
    fig.savefig(PLOTS / filename, dpi=220, bbox_inches="tight")
    plt.close(fig)

def plot_labeled_split(split, data):
    analysis = data["analysis"]
    envelope = analysis["envelope"]
    y = envelope.get("label_raw")
    if y is None:
        pred = envelope["moe_prediction_raw"]
        lo, hi = envelope["expert_min_raw"], envelope["expert_max_raw"]
        fig, ax = plt.subplots(figsize=(8, 5))
        ax.hist(pred, bins=np.arange(-10, 20.5, .25), color="darkorange", alpha=.8)
        ax.set(xlim=(0, 10), xlabel="Raw score interpretation (0–10 display)", ylabel="Sample count",
               title=f"{split}: label-free MOE8 prediction histogram (n={len(pred)}; GT unavailable)")
        save_plot(fig, f"{split}_01_hist_prediction_labels_unavailable.png")
        fig, ax = plt.subplots(figsize=(8, 5)); bins = np.arange(-10, 20.5, .25)
        ax.hist(lo, bins=bins, histtype="step", linewidth=1.8, label="Expert minimum")
        ax.hist(hi, bins=bins, histtype="step", linewidth=1.8, label="Expert maximum")
        ax.set(xlim=(0, 10), xlabel="Raw score interpretation (0–10 display)", ylabel="Count",
               title=f"{split}: expert envelope endpoint distributions (n={len(pred)}; GT unavailable)")
        ax.legend(); save_plot(fig, f"{split}_02_hist_expert_endpoints_labels_unavailable.png")
        fig, ax = plt.subplots(figsize=(8, 5)); ax.hist(envelope["envelope_width_raw"], bins=40, color="slateblue")
        ax.set(xlabel="Expert envelope width (raw units)", ylabel="Sample count",
               title=f"{split}: label-free envelope width (n={len(pred)})")
        save_plot(fig, f"{split}_09_envelope_width_distribution.png")
        return
    y = np.asarray(y)
    pred = envelope["moe_prediction_raw"]
    lo, hi, width = envelope["expert_min_raw"], envelope["expert_max_raw"], envelope["envelope_width_raw"]
    tag = f"{split}_"
    fig, ax = plt.subplots(figsize=(8, 5))
    ax.hist(y, bins=np.arange(0, 10.5, .25), alpha=.55, label=f"Ground truth (n={len(y)})", density=True)
    ax.hist(pred, bins=np.arange(-10, 20.5, .25), alpha=.55, label="Frozen MOE8 prediction", density=True)
    pred_outside = int(np.sum((pred < 0) | (pred > 10)))
    ax.set(xlim=(0, 10), xlabel="Raw score (0–10)", ylabel="Density", title=f"{split}: ground truth vs MOE8 predictions (n={len(y)}; predictions outside display={pred_outside})")
    ax.legend(); save_plot(fig, tag + "01_hist_ground_truth_vs_prediction.png")

    fig, ax = plt.subplots(figsize=(8, 5))
    bins = np.arange(-10, 20.5, .25)
    ax.hist(y, bins=bins, histtype="step", linewidth=2, label="Ground truth")
    ax.hist(lo, bins=bins, histtype="step", linewidth=1.8, label="Expert minimum")
    ax.hist(hi, bins=bins, histtype="step", linewidth=1.8, label="Expert maximum")
    endpoint_outside = int(np.sum((lo < 0) | (lo > 10)) + np.sum((hi < 0) | (hi > 10)))
    ax.set(xlim=(0, 10), xlabel="Raw score units (0–10 display)", ylabel="Count", title=f"{split}: target and expert endpoints (n={len(y)}; endpoint values outside display={endpoint_outside})")
    ax.legend(); save_plot(fig, tag + "02_hist_ground_truth_vs_expert_endpoints.png")

    for values, name, label in ((hi, "03_ground_truth_vs_expert_max.png", "Expert maximum"),
                                (lo, "04_ground_truth_vs_expert_min.png", "Expert minimum")):
        fig, ax = plt.subplots(figsize=(6.5, 6))
        ax.scatter(y, values, s=9, alpha=.35, rasterized=True)
        ax.plot([0, 10], [0, 10], "k--", linewidth=1)
        outside_count = int(np.sum((values < 0) | (values > 10)))
        ax.set(xlim=(0, 10), ylim=(0, 10), xlabel="Ground truth (raw 0–10)", ylabel=f"{label} (raw interpretation)", title=f"{split}: ground truth vs {label.lower()} (n={len(y)}; expert values outside display={outside_count})")
        save_plot(fig, tag + name)

    fig, ax = plt.subplots(figsize=(6.5, 6))
    ax.scatter(y, pred, s=9, alpha=.35, rasterized=True)
    ax.plot([0, 10], [0, 10], "k--", linewidth=1)
    ax.set(xlim=(0, 10), ylim=(0, 10), xlabel="Ground truth (raw 0–10)", ylabel="Frozen MOE8 prediction (raw)", title=f"{split}: prediction vs target (n={len(y)}; prediction values outside display={int(np.sum((pred < 0) | (pred > 10)))})")
    save_plot(fig, tag + "05_ground_truth_vs_moe_prediction.png")

    bins = analysis["per_bin_metrics"]
    labels = [f"{r['ground_truth_bin']}\nn={r['sample_count']}" for r in bins]
    for field, title, filename in (("above_expert_max_fraction", "Ground truth above every expert", "06_fraction_above_expert_max.png"),
                                   ("below_expert_min_fraction", "Ground truth below every expert", "07_fraction_below_expert_min.png")):
        vals = [r[field] for r in bins]
        fig, ax = plt.subplots(figsize=(10, 4.5)); ax.bar(labels, [0 if v is None else v for v in vals])
        ax.set(ylim=(0, 1), ylabel="Fraction", title=f"{split}: {title} by target bin (n={len(y)})")
        ax.tick_params(axis="x", rotation=35); save_plot(fig, tag + filename)

    fig, ax = plt.subplots(figsize=(10, 4.5)); x = np.arange(len(bins)); bw = .38
    ax.bar(x - bw/2, [0 if r["convex_mse_lower_bound"] is None else r["convex_mse_lower_bound"] for r in bins], bw, label="Convex oracle MSE lower bound")
    ax.bar(x + bw/2, [0 if r["actual_moe_mse"] is None else r["actual_moe_mse"] for r in bins], bw, label="Frozen MOE8 MSE")
    ax.set(xticks=x, xticklabels=labels, ylabel="MSE (raw score²)", title=f"{split}: structural lower bound vs gate error (n={len(y)})")
    ax.tick_params(axis="x", rotation=35); ax.legend(); save_plot(fig, tag + "08_per_bin_lower_bound_vs_actual_mse.png")

    fig, ax = plt.subplots(figsize=(8, 5)); ax.hist(width, bins=40, color="slateblue", alpha=.8)
    ax.set(xlabel="Expert envelope width (raw units)", ylabel="Sample count", title=f"{split}: envelope width distribution (n={len(width)})")
    save_plot(fig, tag + "09_envelope_width_distribution.png")

    tails = analysis["tail_metrics"]
    fig, ax = plt.subplots(figsize=(9, 5)); x = np.arange(len(tails)); bw = .25
    for offset, field, label in ((-bw, "outside_envelope_fraction", "Outside envelope"),
                                 (0, "above_expert_max_fraction", "Above max"),
                                 (bw, "below_expert_min_fraction", "Below min")):
        ax.bar(x + offset, [0 if r[field] is None else r[field] for r in tails], bw, label=label)
    ax.set(xticks=x, xticklabels=[f"{r['ground_truth_range']}\nn={r['sample_count']}" for r in tails], ylim=(0, 1), ylabel="Fraction", title=f"{split}: low/middle/high tail envelope diagnostics (n={len(y)})")
    ax.legend(); save_plot(fig, tag + "10_tail_analysis.png")
    out_of_view = int(np.sum((pred < 0) | (pred > 10)) + np.sum((lo < 0) | (lo > 10)) + np.sum((hi < 0) | (hi > 10)))
    return out_of_view

plot_display_outliers = {}
for split, data in split_outputs.items():
    plot_display_outliers[split] = plot_labeled_split(split, data)

## 11. Generator diagnostics, interpretation, and saved provenance

Generator metrics use only official `model_id`/`model_name` joined to the same ordered source rows. This experiment is not an unseen-generator benchmark: several component experts were fit using the complete official training split.

In [ ]:
if generator_available:
    for split, data in split_outputs.items():
        labels = data["analysis"]["envelope"].get("label_raw")
        if labels is None:
            continue
        rows = metadata[split] if split != "test" else test_rows_after_inference
        if not rows:
            continue
        field = ("model_id" if all(r.get("model_id") for r in rows) else
                 "model_name" if all(r.get("model_name") for r in rows) else None)
        if field is None:
            continue
        counts = {}
        for row in rows:
            counts[row[field]] = counts.get(row[field], 0) + 1
        selected = [r for r in generator_rows if r.get("split") == split and r.get("status") == "AVAILABLE"
                    and counts.get(r.get("generator_id"), 0) >= 50]
        if selected:
            fig, ax = plt.subplots(figsize=(max(8, len(selected) * .45), 5))
            ax.bar([f"{r['generator_id']}\nn={r['sample_count']}" for r in selected], [r["outside_envelope_fraction"] for r in selected])
            ax.set(ylim=(0, 1), xlabel="Official generator ID", ylabel="Outside-envelope fraction",
                   title=f"{split}: generator-stratified envelope failure (n≥50 per group)")
            ax.tick_params(axis="x", rotation=50)
            save_plot(fig, f"{split}_generator_outside_envelope.png")

def write_report():
    lines = [
        "# D01 — MOE8: phân tích bao điểm của chuyên gia", "",
        "## Phạm vi và cách đọc", "",
        f"Giữ nguyên toàn bộ {sum(len(v['ids']) for v in split_outputs.values()):,} mẫu public train/validation/test theo thứ tự nguồn; không loại bỏ WAV trùng. Có {D01_PUBLIC_SPLIT_OVERLAP_DIAGNOSTIC['total_overlapping_sha256_groups']} nhóm SHA-256 trùng, với số nhóm theo cặp split: {D01_PUBLIC_SPLIT_OVERLAP_DIAGNOSTIC['pair_overlap_group_counts']}. WAV trùng giữa split có thể làm lệch đánh giá. Đối chiếu loại mẫu trùng chỉ áp dụng khi gộp metric sensitivity tùy chọn và được lưu ở overlap_metric_sensitivity.json.", "",
        f"Đây là chẩn đoán trên gate MOE8 đóng băng, không huấn luyện hay hiệu chỉnh điểm. MSE lower bound là oracle chiếu từng nhãn vào khoảng điểm của tám expert; đây không phải hiệu năng mà một gate học được bảo đảm đạt. Sai số bình phương ở raw 0–10 gấp 25 lần sai số ở normalized −1..1. Nhóm C/D dùng ngưỡng cố định {ACCURACY_THRESHOLD_RAW:.2f} raw point; sensitivity tại {SENSITIVITY_THRESHOLDS_RAW} nằm trong summary.json.", "",
        "E1/E2 mang ngữ nghĩa similarity proxy (CLAP/MGA), trong khi E3–E8 là điểm hồi quy normalized. Quy trình giữ nguyên điểm gốc theo contract; phép đổi `5*s+5` chỉ diễn giải trên raw-unit axis, không phải calibration. Giá trị proxy ngoài 0–10 vẫn được giữ trong CSV và metric; vùng ngoài trục hiển thị được ghi nhận.", "",
        "Bảng scale audit dùng contract thực từ Full8 và thống kê score quan sát theo split; E1/E2 không được xem như nhãn hồi quy chỉ vì cùng tham gia convex fusion.", "",
        "| Expert | semantics | output scale | train min..max normalized | train ngoài [-1,1] |",
        "|---|---|---|---:|---:|",
    ]
    for expert_name in EXPECTED_ORDER:
        audit = score_scale_audit[expert_name]
        train = audit["by_split"]["train"]
        lines.append(f"| {expert_name} | {audit['contract']['semantics']} | {audit['contract']['output_scale']} | {train['normalized_min']:.5f}..{train['normalized_max']:.5f} | {train['fraction_outside_minus1_to1']:.5f} |")
    lines += [
        "",
        "Train là mô tả in-sample; validation phản ánh lịch sử lựa chọn validation của dự án. Đây không phải LOGO và không chứng minh generalization tới generator chưa thấy.", "",
        "## Kết quả theo split", "",
    ]
    for split in EXPECTED_COUNTS:
        analysis = split_outputs[split]["analysis"]
        lines += [f"### {split} (n={len(split_outputs[split]['ids'])})", ""]
        metrics = analysis.get("global_metrics")
        if metrics is None:
            lines += ["Không có nhãn công khai hợp lệ; chỉ xuất phân phối dự đoán, khoảng expert, độ rộng và bất biến gate. Không tính oracle, SRCC hoặc MSE.", ""]
            continue
        high = next(r for r in analysis["per_bin_metrics"] if r["ground_truth_bin"] == "[9,10]")
        classes = analysis["classification"]["class_summary"]
        lines += [
            f"- Bao convex: outside {metrics['outside_envelope_count']}/{metrics['sample_count']} ({metrics['outside_envelope_fraction']:.4f}); trên expert max {metrics['above_expert_max_count']} ({metrics['above_expert_max_fraction']:.4f}), dưới expert min {metrics['below_expert_min_count']} ({metrics['below_expert_min_fraction']:.4f}).",
            f"- MSE raw²: lower bound {metrics['convex_mse_lower_bound_raw']:.6f}; MOE8 {metrics['actual_moe_mse_raw']:.6f}; phần chênh {metrics['actual_moe_mse_minus_lower_bound_raw']:.6f}.",
            f"- Độ rộng envelope trung vị/trung bình: {metrics['median_envelope_width_raw']:.4f}/{metrics['mean_envelope_width_raw']:.4f} raw points; sai số tuyệt đối oracle trung bình {metrics['mean_minimum_absolute_error_raw']:.4f}.",
            f"- Nhãn 9–10: n={high['sample_count']}; cao hơn mọi expert: {high['above_expert_max_count']} ({high['above_expert_max_fraction']}); nằm trong envelope: {high['sample_count']-high['above_expert_max_count']-high['below_expert_min_count']}; nhóm C (trong envelope nhưng gate lệch > {analysis['classification']['accuracy_threshold_raw']:.2f} raw point): {classes['C']['sample_count']}; nhóm D chính xác: {classes['D']['sample_count']}.",
            f"- Phân lớp toàn split A/B/C/D: {', '.join(f'{k}={classes[k]["sample_count"]}' for k in 'ABCD')}. A: target trên tất cả expert; B: dưới tất cả; C: bên trong nhưng gate lệch quá ngưỡng; D: bên trong và gate sai số không vượt ngưỡng.", "",
            "| Vùng nhãn | n | outside | trên max | dưới min | MSE lower bound raw² | MOE8 MSE raw² | độ rộng TB |",
            "|---|---:|---:|---:|---:|---:|---:|---:|",
        ]
        for row in analysis["tail_metrics"]:
            lines.append(f"| {row['ground_truth_range']} | {row['sample_count']} | {row['outside_envelope_fraction']} | {row['above_expert_max_fraction']} | {row['below_expert_min_fraction']} | {row['convex_mse_lower_bound_raw']} | {row['actual_moe_mse_raw']} | {row['mean_envelope_width_raw']} |")
        lines.append("")
    if test_rows_after_inference is None:
        lines += ["## Public test", "", "Không tìm thấy nhãn được phát hành trong metadata public test; không báo cáo ground-truth metrics cho test.", ""]
    else:
        lines += ["## Public test", "", "Nhãn public test chỉ được đọc sau khi xuất dự đoán test label-free và chỉ dùng mô tả hậu nghiệm; không dùng để train, chọn mô hình, fit calibration hoặc oracle predictor.", ""]
    high = next((r for r in split_outputs["validation"]["analysis"]["per_bin_metrics"]
                 if r["ground_truth_bin"] == "[9,10]"), None)
    if high is None or high["sample_count"] == 0:
        conclusion = "Không đủ nhãn 9–10 ở validation để kết luận vùng đuôi cao; cần đọc các bảng theo-bin và không ngoại suy."
    else:
        inside = high["sample_count"] - high["above_expert_max_count"] - high["below_expert_min_count"]
        outside = high["outside_envelope_count"]
        if outside > inside:
            conclusion = (f"Ở validation, {outside}/{high['sample_count']} mẫu 9–10 nằm ngoài envelope "
                          f"(trong đó {high['above_expert_max_count']} ở trên mọi expert và "
                          f"{high['below_expert_min_count']} ở dưới mọi expert); bằng chứng nghiêng về giới hạn cấu trúc, "
                          "nên ưu tiên Direction A để thử score fusion có residual/non-convex.")
        elif inside > high["above_expert_max_count"]:
            conclusion = "Ở validation, nhiều mẫu 9–10 vẫn nằm trong envelope, nên expert có thể tạo điểm cao nếu chọn trọng số thích hợp; sai số gate trong phần đó là cơ hội tiềm năng cho Direction B, nhưng oracle không bảo đảm gate học được mức này."
        else:
            conclusion = "Ở validation, vùng 9–10 có cả mẫu ngoài envelope và mẫu còn khả thi bên trong; bằng chứng phù hợp với hạn chế hỗn hợp. Cần cân nhắc Direction A và phân tích gating cho các mẫu còn trong khoảng."
    lines += ["## Kết luận và hướng ưu tiên", "", conclusion,
              "Mức cấu trúc được đọc qua convex MSE lower bound và số mẫu A/B; cơ hội gating được đọc qua chênh actual MSE − lower bound và nhóm C. Chênh lớn là khoảng trống tiềm năng, không phải bảo đảm cải thiện bởi gate có thể học.", "",
              "## Giới hạn", "", "Gate và expert checkpoints giữ nguyên; không calibration, clamping hay tối ưu lại. Kết quả train có thể in-sample, validation có thể chịu ảnh hưởng selection history, test chỉ báo cáo khi nhãn public đã phát hành. Không có kết luận về private evaluation.", ""]
    (REPORTS / "D01-MOE8-report.md").write_text("\n".join(lines), encoding="utf-8")

write_report()
provenance = {
    "source_commit": SOURCE_COMMIT, "source_repo": REPO_URL,
    "dataset_ids": DATASET_IDS, "private_evaluation_attached": False,
    "official_dataset_root": str(DATA_ROOT), "gate_checkpoint": str(gate_checkpoint),
    "gate_checkpoint_sha256": gate_sha256, "feature_store_root": str(FULL8_ROOT),
    "profile": FULL8_MANIFESTS["train"]["expert_profile"],
    "expert_order": FULL8_MANIFESTS["train"]["expert_order"],
    "feature_width": FULL8_MANIFESTS["train"]["feature_width"],
    "manifests": {s: {"store_identity_sha256": FULL8_MANIFESTS[s]["store_identity_sha256"],
                       "source_manifest_sha256": FULL8_MANIFESTS[s]["source_manifest_sha256"],
                       "sample_count": sum(x["count"] for x in FULL8_MANIFESTS[s]["shards"]),
                       "labels_present": FULL8_MANIFESTS[s]["labels_present"],
                       "score_contracts": FULL8_MANIFESTS[s].get("assembly_identity", {}).get("score_contracts")}
                  for s in EXPECTED_COUNTS},
    "expert8_cache_identity": FULL8_MANIFESTS["train"]["assembly_identity"].get("expert8_cache_identity"),
    "gate_format_version": GATE_PAYLOAD.get("format_version"),
    "gate_architecture": GATE_PAYLOAD.get("gate_architecture"),
    "gate_source_code_sha256": GATE_PAYLOAD.get("source_code_sha256"),
    "gate_source_code_audit": GATE_SOURCE_CODE_AUDIT,
    "train_validation_overlap": EXPECTED_OVERLAP_DIAGNOSTIC,
    "train_validation_overlap_policy_origin": GATE_OVERLAP_POLICY_ORIGIN,
    "checkpoint_expert_provenance": GATE_PAYLOAD.get("expert_checkpoint_provenance"),
    "expert_score_scale_audit": score_scale_audit,
    "score_scale_warning": "E1/E2 similarity proxies retained without calibration; E3-E8 regression-normalized.",
    "test_labels_read_after_inference": test_rows_after_inference is not None,
    "plot_display_out_of_range_counts": plot_display_outliers,
    "normalization": "raw=(normalized*5)+5; raw MSE = normalized MSE * 25",
}
(REPORTS / "provenance.json").write_text(json.dumps(provenance, indent=2, allow_nan=False) + "\n")
print("Artifacts:", WORK)
print("Plots:", len(list(PLOTS.glob("*.png"))))